In [3]:
"""
NQ IB / ORB / Order-Flow Pipeline -- v5
Stage 1: Load, timezone-normalize, and combine the seven 1-year tick files
into a single 1-min OHLCV + order-flow bar dataset (2020-2026).

Run this once. It caches the combined 1-min bars to parquet so later stages
(daily feature engineering, signal scoring, analysis) load that cache instead
of re-reading raw ticks every time.
"""

import pandas as pd
import numpy as np
import pyarrow.parquet as pq
from pathlib import Path
import gc
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CONFIG
# ============================================================

DATA_DIR = Path(r"C:\Users\tammm\NQ and ES analysis")

TICK_FILES = [
    DATA_DIR / "NQ_ticks_2020.parquet",
    DATA_DIR / "NQ_ticks_2021.parquet",
    DATA_DIR / "NQ_ticks_2022.parquet",
    DATA_DIR / "NQ_ticks_2023.parquet",
    DATA_DIR / "NQ_ticks_2024.parquet",
    DATA_DIR / "NQ_ticks_2025.parquet",
    DATA_DIR / "NQ_ticks_2026.parquet",
]

# Set based on what you see in the TIMEZONE / COVERAGE SANITY CHECK printout
# the first time you run this:
#   'utc'        -> raw timestamps are UTC (or tz-aware in another zone) and
#                   need tz_convert to US/Eastern  [matches the original
#                   order-flow script's assumption -- start here]
#   'us/eastern' -> raw timestamps are already naive US/Eastern wall-clock
#                   times (just need tz_localize, no shift)
SOURCE_TZ_MODE = 'utc'

OUTPUT_BARS_PATH = DATA_DIR / "nq_combined_1min_bars_2020_2026.parquet"

# ============================================================
# STEP 1 -- per-file load + tz normalize + per-tick delta
# ============================================================

NEEDED_COLUMNS = ['timestamp', 'price', 'volume', 'aggressor']


def load_tick_file(path: Path) -> pd.DataFrame:
    """Load one year's tick parquet, normalize timestamp to US/Eastern,
    and compute per-tick buy/sell volume and delta from the aggressor tag.

    Memory notes (these files run ~150M rows/year):
      - Only the columns we actually use are read off disk.
      - 'aggressor' is cleaned via its (small) set of category labels, not
        via .str ops over every row -- .str.strip()/.str.capitalize() on a
        150M-row object column creates several full-length temporary arrays
        and is what caused the earlier MemoryError.
      - buy_vol/sell_vol/delta are built from raw numpy arrays, not extra
        pandas Series, to avoid further copies.
    """

    schema_cols = pq.ParquetFile(path).schema_arrow.names
    read_cols = [c for c in NEEDED_COLUMNS if c in schema_cols] or None
    ticks = pd.read_parquet(path, columns=read_cols)

    # --- timestamp handling ---
    # Always work with a DatetimeIndex (not a Series) here -- a Series needs
    # the .dt accessor for .tz/.tz_localize/.tz_convert, a DatetimeIndex has
    # them directly, and using an Index throughout avoids branching later.
    has_ts_col = 'timestamp' in ticks.columns
    raw_ts = pd.DatetimeIndex(pd.to_datetime(ticks['timestamp'] if has_ts_col else ticks.index))
    if has_ts_col:
        ticks = ticks.drop(columns=['timestamp'])

    if SOURCE_TZ_MODE == 'utc':
        if raw_ts.tz is None:
            raw_ts = raw_ts.tz_localize('UTC')
        ts_eastern = raw_ts.tz_convert('US/Eastern')
    elif SOURCE_TZ_MODE == 'us/eastern':
        ts_eastern = raw_ts.tz_localize('US/Eastern') if raw_ts.tz is None else raw_ts.tz_convert('US/Eastern')
    else:
        raise ValueError(f"Unknown SOURCE_TZ_MODE: {SOURCE_TZ_MODE!r}")

    ticks.index = pd.DatetimeIndex(ts_eastern, name='timestamp')
    ticks = ticks.sort_index()

    # --- aggressor-based delta (categorical, not per-row .str ops) ---
    agg_cat = ticks['aggressor'].astype('category')
    cleaned_categories = agg_cat.cat.categories.str.strip().str.capitalize()
    agg_cat = agg_cat.cat.rename_categories(cleaned_categories)

    buy_mask  = (agg_cat == 'Buy').to_numpy()
    sell_mask = (agg_cat == 'Sell').to_numpy()
    volume    = ticks['volume'].to_numpy()

    ticks['buy_vol']  = np.where(buy_mask, volume, 0)
    ticks['sell_vol'] = np.where(sell_mask, volume, 0)
    ticks['delta']    = ticks['buy_vol'] - ticks['sell_vol']
    ticks = ticks.drop(columns=['aggressor'])  # no longer needed past this point

    del agg_cat, buy_mask, sell_mask, volume
    gc.collect()

    return ticks


def resample_to_1min(ticks: pd.DataFrame) -> pd.DataFrame:
    """Resample tick data to 1-min OHLCV + order-flow bars."""
    bars = ticks.resample('1min').agg(
        open=('price', 'first'),
        high=('price', 'max'),
        low=('price', 'min'),
        close=('price', 'last'),
        volume=('volume', 'sum'),
        buy_vol=('buy_vol', 'sum'),
        sell_vol=('sell_vol', 'sum'),
        delta=('delta', 'sum'),
        trades=('price', 'count'),
    ).dropna(subset=['open'])
    return bars


def process_one_file(path: Path) -> pd.DataFrame:
    print(f"  Loading {path.name} ...")
    ticks = load_tick_file(path)
    n_ticks = len(ticks)
    date_min, date_max = ticks.index.min(), ticks.index.max()
    bars = resample_to_1min(ticks)
    del ticks  # free raw tick memory before moving to the next file
    gc.collect()
    print(f"    {n_ticks:,} ticks  |  {date_min} -> {date_max}  |  {len(bars):,} 1-min bars")
    return bars


# ============================================================
# STEP 2 -- combine all seven years
# ============================================================

def build_combined_bars() -> pd.DataFrame:
    print("Processing tick files one year at a time (raw ticks discarded after each resample)...\n")
    yearly_bars = [process_one_file(p) for p in TICK_FILES]

    combined = pd.concat(yearly_bars).sort_index()
    before = len(combined)
    combined = combined[~combined.index.duplicated(keep='first')]
    after = len(combined)
    if before != after:
        print(f"\n  Dropped {before - after:,} duplicate-timestamp bars at file boundaries.")

    return combined


def sanity_check(bars: pd.DataFrame):
    print("\n" + "=" * 70)
    print("TIMEZONE / COVERAGE SANITY CHECK")
    print("=" * 70)
    print(f"  Combined bars       : {len(bars):,}")
    print(f"  Date range          : {bars.index.min()}  ->  {bars.index.max()}")

    # Pick one arbitrary mid-sample weekday and show the bar(s) covering
    # 9:30am ET. If SOURCE_TZ_MODE is correct, this should look like a
    # normal, liquid NQ RTH open (price plausible, volume/trades elevated
    # relative to overnight bars).
    all_days = bars.index.normalize().unique()
    weekdays = [d for d in all_days if d.weekday() < 5]
    sample_day = weekdays[len(weekdays) // 2]
    probe = bars.loc[
        sample_day.strftime('%Y-%m-%d') + ' 09:30:00':
        sample_day.strftime('%Y-%m-%d') + ' 09:35:00'
    ]
    print(f"\n  Sample RTH-open window on {sample_day.date()} (should be ~9:30am ET, "
          f"liquid NQ price/volume):")
    print(probe)
    print("\n  If this does NOT look like a normal 9:30am NQ open, flip SOURCE_TZ_MODE")
    print("  at the top of this file and re-run.")

    print("\n  Per-year bar counts (check for gaps at file-seam years):")
    print(bars.groupby(bars.index.year).size())


if __name__ == '__main__':
    combined_bars = build_combined_bars()
    sanity_check(combined_bars)

    combined_bars.to_parquet(OUTPUT_BARS_PATH)
    print(f"\nSaved combined 1-min bars -> {OUTPUT_BARS_PATH}")
    print("This is the file Stage 2 (daily feature engineering: ORB-15/ORB-30,")
    print("IB structure, order-flow signals, session context) will load.")

Processing tick files one year at a time (raw ticks discarded after each resample)...

  Loading NQ_ticks_2020.parquet ...
    116,975,063 ticks  |  2020-01-01 18:00:00.017000-05:00 -> 2020-12-31 16:59:59.720412504-05:00  |  349,463 1-min bars
  Loading NQ_ticks_2021.parquet ...
    117,940,868 ticks  |  2021-01-03 18:00:00.065000-05:00 -> 2021-12-31 16:59:59.979397239-05:00  |  353,413 1-min bars
  Loading NQ_ticks_2022.parquet ...
    149,419,593 ticks  |  2022-01-02 18:00:00.076000-05:00 -> 2022-12-30 16:59:59.582000-05:00  |  354,030 1-min bars
  Loading NQ_ticks_2023.parquet ...
    141,545,088 ticks  |  2023-01-02 18:00:00.083000-05:00 -> 2023-12-29 16:59:58.065000-05:00  |  353,402 1-min bars
  Loading NQ_ticks_2024.parquet ...
    135,556,040 ticks  |  2024-01-01 18:00:00.056000-05:00 -> 2024-12-31 16:59:59.000884126-05:00  |  355,006 1-min bars
  Loading NQ_ticks_2025.parquet ...
    127,583,875 ticks  |  2025-01-01 18:00:00.027000-05:00 -> 2025-12-31 16:59:59.102453-05:00  | 

In [20]:
"""
NQ IB / ORB / Order-Flow Pipeline -- v5
Stage 2: build the unified per-day dataset (2020-2026) from the cached
1-min bars produced by Stage 1.

Computes, per trading day:
  - Session context: Asia/London direction, 4-way structural pattern,
    NY-open-vs-London-midpoint, NY-open-vs-overnight-midpoint
  - ORB-15 (9:30-9:45): close half, direction, position in IB, curve result
  - ORB-30 (9:30-10:00): close half, direction, position in IB, TWO curve
    variants (within-IB and post-IB)
  - IB structure: OHLC, range, midpoint, direction, close half/position,
    extreme-first, quarter/curve type
  - Order flow: IB delta, delta POC (+zone), delta-by-quartile structure
    label, four pre-IB delta lookback windows, delta over the ORB-15/ORB-30
    windows themselves
  - Chain agreement flags (ORB close-half vs IB close-half, extreme-first
    vs close-half)
  - Outcomes: post-IB direction, first post-IB break direction, 1h/session
    break flags
  - Structural score (10-signal vote) + order-flow score (4-signal vote) +
    a boolean confirmation flag + trade type (extension/fade)

NOTE on scope: this stage does NOT yet include MFE/MAE-after-break or
midpoint-retest fields (present in the older price-only script) -- easy to
add later if you want them, just not built here to keep this stage focused.
"""

import pandas as pd
import numpy as np
from scipy import stats as scipy_stats
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CONFIG
# ============================================================

DATA_DIR = Path(r"C:\Users\tammm\NQ and ES analysis")
BARS_PATH = DATA_DIR / "nq_combined_1min_bars_2020_2026.parquet"
OUTPUT_DATASET_PATH = DATA_DIR / "nq_v5_unified_dataset.csv"

MIN_ORB_RANGE_FRACTION = 0.03     # ORB range must be >= this fraction of
                                   # the previous full session's range to
                                   # count as "meaningful"
MIN_STRUCTURAL_AGREEMENT = 6      # out of 10 structural signals
MIN_ORDERFLOW_AGREEMENT  = 3      # out of 4 order-flow signals

PRE_IB_WINDOWS = {
    'pre_ib_0600_0930': ('06:00', '09:30'),
    'pre_ib_0800_0930': ('08:00', '09:30'),
    'pre_ib_0830_0930': ('08:30', '09:30'),
    'pre_ib_0900_0930': ('09:00', '09:30'),
}

# ============================================================
# HELPERS -- structural
# ============================================================

def close_half(close, high, low):
    if high == low:
        return None
    mid = (high + low) / 2
    return 'upper' if close >= mid else 'lower'


def position_in_range(price, high, low):
    if high == low:
        return None
    return (price - low) / (high - low)


def position_bucket(score):
    if score is None:
        return None
    if score <= 1 / 3:
        return 'bottom_third'
    if score <= 2 / 3:
        return 'middle_third'
    return 'top_third'


def zone_label(pct):
    if pct < 0.25:
        return 'Q1'
    if pct < 0.50:
        return 'Q2'
    if pct < 0.75:
        return 'Q3'
    return 'Q4'


def session_direction(o, c):
    if o == c:
        return None
    return 'bullish' if c > o else 'bearish'


def classify_asia_london(london_high, london_low, asia_high, asia_low):
    """Four-way exhaustive taxonomy (matches the v4 script)."""
    if london_high > asia_high and london_low < asia_low:
        return 'london_engulfs_asia'
    elif london_high <= asia_high and london_low >= asia_low:
        return 'asia_engulfs_london'
    elif london_high > asia_high and london_low >= asia_low:
        return 'london_partially_engulfs_upwards'
    elif london_low < asia_low and london_high <= asia_high:
        return 'london_partially_engulfs_downwards'
    else:
        return 'other'


def which_quarter(ts, q1_end, q2_end, q3_end):
    if ts < q1_end:
        return 1
    if ts < q2_end:
        return 2
    if ts < q3_end:
        return 3
    return 4


def which_extreme_first(data, ref_high, ref_low):
    """Scan a slice chronologically and report whether ref_high or ref_low
    was touched first. Used both for the IB's final extreme-first (which has
    look-ahead built in if the true extreme ends up set in Q3/Q4) and for
    each ORB's OWN internal extreme-first (fully known the moment that ORB's
    window closes -- no look-ahead)."""
    for _, row in data.iterrows():
        hit_high = row['high'] >= ref_high
        hit_low = row['low'] <= ref_low
        if hit_high and hit_low:
            return 'simultaneous'
        elif hit_high:
            return 'high'
        elif hit_low:
            return 'low'
    return None


# Skew threshold (percentage points of volume) beyond which a window is
# called directional (P/b) rather than checked for balance (D) vs bimodal
# (B). This is a tunable heuristic, not a fixed standard -- adjust if the
# shape distribution in Stage 4 looks off.
PROFILE_SKEW_THRESHOLD = 15.0


def quartile_profile(data, ref_high, ref_low):
    """Bin a window's 1-min bars into 4 equal price bands of its OWN range
    and return volume/delta sums + percentages per quartile, plus a P/b/B/D
    shape label:
      P -- top-heavy, thin bottom tail   (short-covering / breakout-up)
      b -- bottom-heavy, thin top tail   (long-liquidation / breakout-down)
      D -- middle-heavy, balanced        (rotational, two-sided / normal day)
      B -- extremes-heavy, thin middle   (bimodal / double distribution)
    Returns None if the window is empty or has zero range/volume.
    """
    ref_range = ref_high - ref_low
    if ref_range == 0 or data.empty:
        return None

    dcp = data.copy()
    dcp['price_pos'] = ((dcp['high'] + dcp['low']) / 2 - ref_low) / ref_range

    q_bounds = [(0, 0.25), (0.25, 0.50), (0.50, 0.75), (0.75, 1.0001)]
    vols, deltas = [], []
    for qlo, qhi in q_bounds:
        m = (dcp['price_pos'] >= qlo) & (dcp['price_pos'] < qhi)
        vols.append(dcp.loc[m, 'volume'].sum())
        deltas.append(dcp.loc[m, 'delta'].sum())

    total_vol = sum(vols)
    if total_vol == 0:
        return None
    v_pct = [v / total_vol * 100 for v in vols]

    top_half = v_pct[2] + v_pct[3]
    bottom_half = v_pct[0] + v_pct[1]
    middle = v_pct[1] + v_pct[2]
    extremes = v_pct[0] + v_pct[3]
    skew = top_half - bottom_half

    if skew > PROFILE_SKEW_THRESHOLD:
        shape = 'P'
    elif skew < -PROFILE_SKEW_THRESHOLD:
        shape = 'b'
    elif middle > extremes:
        shape = 'D'
    else:
        shape = 'B'

    return {
        'v1': vols[0], 'v2': vols[1], 'v3': vols[2], 'v4': vols[3],
        'd1': deltas[0], 'd2': deltas[1], 'd3': deltas[2], 'd4': deltas[3],
        'v1_pct': v_pct[0], 'v2_pct': v_pct[1], 'v3_pct': v_pct[2], 'v4_pct': v_pct[3],
        'shape': shape,
    }


def find_first_break(data, ref_high, ref_low):
    h_broken = l_broken = False
    h_time = l_time = None
    for idx, row in data.iterrows():
        if not h_broken and row['high'] > ref_high:
            h_broken, h_time = True, idx
        if not l_broken and row['low'] < ref_low:
            l_broken, l_time = True, idx
    return h_broken, h_time, l_broken, l_time


def compute_curve_result(breaking_high, breaking_low, ref_high, ref_low,
                          sh_high, sh_low):
    """Generic curve-vs-continuation classifier, factored out so it can
    serve ORB-15 (breaking block=Q2, second-half ref=Q3+Q4), ORB-30
    within-IB (breaking block=Q3, second-half ref=Q4 alone), and ORB-30
    post-IB (breaking block=Q3, second-half ref=full post-IB session).

    breaking_high/low = high/low of the block that might break the range.
    ref_high/low      = the ORB range being tested.
    sh_high/low       = high/low of whatever 'second half' window judges
                        curve vs. continuation.
    """
    breaks_high = breaking_high > ref_high
    breaks_low = breaking_low < ref_low
    if breaks_high and not breaks_low:
        if sh_low < ref_low:
            return 'broke_high_curved_low'
        elif sh_high > breaking_high:
            return 'broke_high_continued_high'
        else:
            return 'broke_high_neither'
    elif breaks_low and not breaks_high:
        if sh_high > ref_high:
            return 'broke_low_curved_high'
        elif sh_low < breaking_low:
            return 'broke_low_continued_low'
        else:
            return 'broke_low_neither'
    elif breaks_high and breaks_low:
        return 'breaks_both'
    else:
        return 'contained'


def delta_structure_label(dq1, dq2, dq3, dq4, ib_close_position):
    """Ported from the original order-flow script's quartile-delta
    classification."""
    if ib_close_position == 'above_midpoint':
        absorbed_at_low = dq1 < 0
        confirmed_at_high = (dq3 > 0) or (dq4 > 0)
        if absorbed_at_low and confirmed_at_high:
            return 'ideal_bullish'
        if confirmed_at_high and not absorbed_at_low:
            return 'buying_only'
        if absorbed_at_low and not confirmed_at_high:
            return 'absorbed_no_confirm'
        return 'weak_bullish'
    else:
        absorbed_at_high = dq4 > 0
        confirmed_at_low = (dq1 < 0) or (dq2 < 0)
        if absorbed_at_high and confirmed_at_low:
            return 'ideal_bearish'
        if confirmed_at_low and not absorbed_at_high:
            return 'selling_only'
        if absorbed_at_high and not confirmed_at_low:
            return 'absorbed_no_confirm'
        return 'weak_bearish'


def compute_trade_type(ib_extreme, ib_close_h):
    if ib_extreme is None or ib_close_h is None:
        return 'unclear'
    if ib_extreme == 'low' and ib_close_h == 'upper':
        return 'extension_long'
    if ib_extreme == 'high' and ib_close_h == 'lower':
        return 'extension_short'
    if ib_extreme == 'high' and ib_close_h == 'upper':
        return 'fade_long'
    if ib_extreme == 'low' and ib_close_h == 'lower':
        return 'fade_short'
    return 'unclear'


# ============================================================
# HELPERS -- signal scoring
# ============================================================

ASIA_LONDON_BIAS = {
    'london_engulfs_asia': 'neutral',
    'asia_engulfs_london': 'neutral',
    'london_partially_engulfs_upwards': 'bullish',
    'london_partially_engulfs_downwards': 'bearish',
}


def score_dir(d):
    if d == 'bullish':
        return 'bullish'
    if d == 'bearish':
        return 'bearish'
    return 'neutral'


def score_pattern(p):
    return ASIA_LONDON_BIAS.get(p, 'neutral')


def score_close_half(ch):
    if ch == 'upper':
        return 'bullish'
    if ch == 'lower':
        return 'bearish'
    return 'neutral'


def score_position(bucket):
    if bucket == 'bottom_third':
        return 'bullish'
    if bucket == 'top_third':
        return 'bearish'
    return 'neutral'


def score_extreme_first(e):
    if e == 'low':
        return 'bullish'
    if e == 'high':
        return 'bearish'
    return 'neutral'


def score_ny_vs_mid(pos):
    if pos == 'above':
        return 'bullish'
    if pos == 'below':
        return 'bearish'
    return 'neutral'


def aggregate(scores, min_agree):
    bull = scores.count('bullish')
    bear = scores.count('bearish')
    if bull == 0 and bear == 0:
        return 'no_edge', bull, bear, False
    bias = 'bullish' if bull >= bear else 'bearish'
    return bias, bull, bear, (max(bull, bear) >= min_agree)


# Order-flow-specific scoring -----------------------------------------------

# NOTE: weak_bullish/weak_bearish mean price closed one way but the delta
# quartile pattern actively contradicted it (e.g. buying at the lows +
# selling at the highs despite a bullish close). Scored here as a signal in
# the OPPOSITE direction (a divergence/fade warning), not as neutral. If you
# read these as "no real signal" rather than "the opposite signal," change
# the two lines below to 'neutral'.
DELTA_STRUCTURE_BIAS = {
    'ideal_bullish': 'bullish', 'buying_only': 'bullish',
    'absorbed_no_confirm': 'neutral',
    'weak_bullish': 'bearish',
    'ideal_bearish': 'bearish', 'selling_only': 'bearish',
    'weak_bearish': 'bullish',
}


def score_delta_bias(bias):
    if bias == 'buy':
        return 'bullish'
    if bias == 'sell':
        return 'bearish'
    return 'neutral'


def score_delta_structure(label):
    return DELTA_STRUCTURE_BIAS.get(label, 'neutral')


def score_poc_alignment(close_vs_poc):
    if close_vs_poc == 'above':
        return 'bullish'
    if close_vs_poc == 'below':
        return 'bearish'
    return 'neutral'


def score_pre_ib_majority(biases):
    clean = [b for b in biases if b is not None]
    if not clean:
        return 'neutral'
    buy = clean.count('buy')
    sell = clean.count('sell')
    if buy > sell:
        return 'bullish'
    if sell > buy:
        return 'bearish'
    return 'neutral'


# ============================================================
# LOAD BARS
# ============================================================

print("Loading combined 1-min bars...")
bars = pd.read_parquet(BARS_PATH)

# Strip tz after Stage 1's conversion to US/Eastern. Everything from here on
# is naive-Eastern wall-clock arithmetic -- this matches how the underlying
# original scripts worked and avoids DST-transition surprises from doing
# Timedelta math on a tz-aware index twice a year.
bars.index = bars.index.tz_localize(None)
bars = bars.sort_index()
print(f"  {len(bars):,} bars | {bars.index.min()} -> {bars.index.max()}")

# ============================================================
# VWAP PREPROCESSING
# ============================================================
# Dollar-volume approximated per bar from typical price (H+L+C)/3 * volume --
# standard practice for intraday VWAP off 1-min bars; a tiny approximation
# vs. true tick-level price*volume, negligible at 1-min granularity.

bars['typical_price'] = (bars['high'] + bars['low'] + bars['close']) / 3
bars['pv'] = bars['typical_price'] * bars['volume']

# ---- Weekly anchor: resets Sunday 18:00 ET, runs through the week ----
dow = bars.index.dayofweek  # Monday=0 ... Sunday=6
days_since_sunday = (dow + 1) % 7
candidate = pd.DatetimeIndex(bars.index.normalize() - pd.to_timedelta(days_since_sunday, unit='D')
                              + pd.Timedelta(hours=18))
needs_adjust = candidate > bars.index
bars['week_start'] = pd.DatetimeIndex(np.where(needs_adjust, candidate - pd.Timedelta(days=7), candidate))

bars['week_cum_pv'] = bars.groupby('week_start')['pv'].cumsum()
bars['week_cum_vol'] = bars.groupby('week_start')['volume'].cumsum()
bars['vwap_weekly'] = bars['week_cum_pv'] / bars['week_cum_vol']

# Whole-week VWAP summary, used for the "previous week" static anchor
week_summary = bars.groupby('week_start').agg(wk_pv=('pv', 'sum'), wk_vol=('volume', 'sum'))
week_summary['wk_vwap'] = week_summary['wk_pv'] / week_summary['wk_vol']
WEEK_VWAP_LOOKUP = week_summary['wk_vwap'].to_dict()

# ---- Session anchor: resets 9:30 ET each day, runs through the day ----
hm = bars.index.hour * 60 + bars.index.minute
before_930 = hm < (9 * 60 + 30)
sess_date = np.where(before_930, bars.index.normalize() - pd.Timedelta(days=1), bars.index.normalize())
bars['session_start_930'] = pd.DatetimeIndex(sess_date) + pd.Timedelta(hours=9, minutes=30)

bars['day_cum_pv'] = bars.groupby('session_start_930')['pv'].cumsum()
bars['day_cum_vol'] = bars.groupby('session_start_930')['volume'].cumsum()
bars['vwap_session'] = bars['day_cum_pv'] / bars['day_cum_vol']

print(f"  VWAP preprocessing done: {bars['week_start'].nunique()} weeks, "
      f"{bars['session_start_930'].nunique()} sessions")


def week_start_for(ts):
    """Scalar version of the weekly-anchor calc above, for per-day lookups
    (e.g. finding last week's stored VWAP) inside the main loop."""
    dow_ = ts.dayofweek
    days_since_sunday_ = (dow_ + 1) % 7
    cand = ts.normalize() - pd.Timedelta(days=int(days_since_sunday_)) + pd.Timedelta(hours=18)
    if cand > ts:
        cand -= pd.Timedelta(days=7)
    return cand


def vs_level(price, level):
    if pd.isna(level):
        return None
    return 'above' if price > level else 'below'

# ============================================================
# TRADING DAY ENUMERATION  (date = calendar date of RTH)
# ============================================================

shifted = (bars.index - pd.Timedelta(hours=7)).normalize()
trading_days = sorted(pd.DatetimeIndex(shifted).unique())
print(f"  {len(trading_days)} candidate trading days")

# ============================================================
# MAIN LOOP
# ============================================================

records = []
skipped = 0
prev_range = None  # carried from the previous successfully-processed day

for i, d in enumerate(trading_days[1:], 1):  # skip first (likely partial) day
    if i % 250 == 0:
        print(f"  Day {i} / {len(trading_days) - 1} ...")

    actual = pd.Timestamp(d) + pd.Timedelta(hours=7)

    asia_start = actual.replace(hour=18) - pd.Timedelta(days=1)
    asia_end = actual.replace(hour=2)
    lon_start = actual.replace(hour=2)
    lon_end = actual.replace(hour=8)

    ib_start = actual.replace(hour=9, minute=30)
    q1_end = actual.replace(hour=9, minute=45)   # ORB-15 end
    q2_end = actual.replace(hour=10, minute=0)   # ORB-30 end
    q3_end = actual.replace(hour=10, minute=15)
    ib_end = actual.replace(hour=10, minute=30)
    ny_end = actual.replace(hour=16, minute=0)

    asia_data = bars.loc[asia_start:asia_end]
    lon_data = bars.loc[lon_start:lon_end]
    orb15_data = bars.loc[ib_start:q1_end]
    orb30_data = bars.loc[ib_start:q2_end]
    q2_data = bars.loc[q1_end:q2_end]
    q3_data = bars.loc[q2_end:q3_end]
    q4_data = bars.loc[q3_end:ib_end]
    ib_data = bars.loc[ib_start:ib_end]
    post_ib_data = bars.loc[ib_end:ny_end]
    sh_after_orb15 = bars.loc[q2_end:ib_end]        # Q3+Q4, for ORB-15 curve
    full_day_data = bars.loc[asia_start:ny_end]     # seeds next day's prev_range

    if any(x.empty for x in [asia_data, lon_data, orb15_data, orb30_data,
                              q2_data, q3_data, q4_data, ib_data, post_ib_data]):
        skipped += 1
        continue
    if len(orb15_data) < 10:  # need most of the 15 one-minute bars present
        skipped += 1
        continue

    # ---- ORB-15 / ORB-30 raw OHLC ----
    orb15_open, orb15_close = orb15_data['open'].iloc[0], orb15_data['close'].iloc[-1]
    orb15_high, orb15_low = orb15_data['high'].max(), orb15_data['low'].min()
    orb15_range = orb15_high - orb15_low
    if pd.isna([orb15_open, orb15_close, orb15_high, orb15_low]).any():
        skipped += 1
        continue

    orb30_open, orb30_close = orb30_data['open'].iloc[0], orb30_data['close'].iloc[-1]
    orb30_high, orb30_low = orb30_data['high'].max(), orb30_data['low'].min()
    orb30_range = orb30_high - orb30_low
    if pd.isna([orb30_open, orb30_close, orb30_high, orb30_low]).any():
        skipped += 1
        continue

    # ---- ORB-15 / ORB-30 volume + buy/sell split + range as % of price ----
    orb15_volume = orb15_data['volume'].sum()
    orb15_buy_vol = orb15_data['buy_vol'].sum()
    orb15_sell_vol = orb15_data['sell_vol'].sum()
    orb15_range_pct = (orb15_range / orb15_open) * 100 if orb15_open else np.nan

    orb30_volume = orb30_data['volume'].sum()
    orb30_buy_vol = orb30_data['buy_vol'].sum()
    orb30_sell_vol = orb30_data['sell_vol'].sum()
    orb30_range_pct = (orb30_range / orb30_open) * 100 if orb30_open else np.nan

    # ---- ORB-15 / ORB-30 quartile volume/delta profile + P/b/B/D shape ----
    orb15_profile = quartile_profile(orb15_data, orb15_high, orb15_low)
    orb30_profile = quartile_profile(orb30_data, orb30_high, orb30_low)

    # ---- IB structure ----
    ib_open, ib_close = ib_data['open'].iloc[0], ib_data['close'].iloc[-1]
    ib_high, ib_low = ib_data['high'].max(), ib_data['low'].min()
    ib_range = ib_high - ib_low
    if pd.isna([ib_open, ib_close, ib_high, ib_low]).any() or ib_range == 0:
        skipped += 1
        continue

    ib_mid = (ib_high + ib_low) / 2
    ib_direction = 'bullish' if ib_close > ib_open else ('bearish' if ib_close < ib_open else None)
    if ib_direction is None:
        skipped += 1
        continue
    ib_close_position = 'above_midpoint' if ib_close > ib_mid else 'below_midpoint'
    ib_close_half_val = close_half(ib_close, ib_high, ib_low)
    ib_close_pct = position_in_range(ib_close, ib_high, ib_low)
    ib_close_zone = zone_label(ib_close_pct)
    ib_range_pct = (ib_range / ib_open) * 100 if ib_open else np.nan
    ib_buy_vol = ib_data['buy_vol'].sum()
    ib_sell_vol = ib_data['sell_vol'].sum()
    ib_profile = quartile_profile(ib_data, ib_high, ib_low)

    # ---- VWAP anchors ----
    current_week_start = week_start_for(actual)
    prev_week_start = current_week_start - pd.Timedelta(days=7)
    vwap_prevweek = WEEK_VWAP_LOOKUP.get(prev_week_start, np.nan)

    premarket_data = bars.loc[actual.replace(hour=6): actual.replace(hour=9)]
    if not premarket_data.empty and premarket_data['volume'].sum() > 0:
        vwap_premarket = premarket_data['pv'].sum() / premarket_data['volume'].sum()
        premarket_open = premarket_data['open'].iloc[0]
        premarket_close = premarket_data['close'].iloc[-1]
        premarket_high = premarket_data['high'].max()
        premarket_low = premarket_data['low'].min()
        premarket_range = premarket_high - premarket_low
        premarket_range_pct = (premarket_range / premarket_open) * 100 if premarket_open else np.nan
        premarket_direction = session_direction(premarket_open, premarket_close)
        premarket_close_half = close_half(premarket_close, premarket_high, premarket_low)
        premarket_volume = premarket_data['volume'].sum()
        premarket_buy_vol = premarket_data['buy_vol'].sum()
        premarket_sell_vol = premarket_data['sell_vol'].sum()
        premarket_delta = premarket_data['delta'].sum()
        premarket_delta_bias = 'buy' if premarket_delta > 0 else 'sell'
        premarket_profile = quartile_profile(premarket_data, premarket_high, premarket_low)
    else:
        vwap_premarket = np.nan
        premarket_open = premarket_close = premarket_high = premarket_low = np.nan
        premarket_range = premarket_range_pct = np.nan
        premarket_direction = None
        premarket_close_half = None
        premarket_volume = premarket_buy_vol = premarket_sell_vol = np.nan
        premarket_delta = np.nan
        premarket_delta_bias = None
        premarket_profile = None

    vwap_weekly_at_orb15 = orb15_data['vwap_weekly'].iloc[-1]
    vwap_weekly_at_orb30 = orb30_data['vwap_weekly'].iloc[-1]
    vwap_weekly_at_ib = ib_data['vwap_weekly'].iloc[-1]

    vwap_session_at_orb15 = orb15_data['vwap_session'].iloc[-1]
    vwap_session_at_orb30 = orb30_data['vwap_session'].iloc[-1]
    vwap_session_at_ib = ib_data['vwap_session'].iloc[-1]

    orb15_vs_vwap_weekly = vs_level(orb15_close, vwap_weekly_at_orb15)
    orb30_vs_vwap_weekly = vs_level(orb30_close, vwap_weekly_at_orb30)
    ib_vs_vwap_weekly = vs_level(ib_close, vwap_weekly_at_ib)

    orb15_vs_vwap_prevweek = vs_level(orb15_close, vwap_prevweek)
    orb30_vs_vwap_prevweek = vs_level(orb30_close, vwap_prevweek)
    ib_vs_vwap_prevweek = vs_level(ib_close, vwap_prevweek)

    orb15_vs_vwap_premarket = vs_level(orb15_close, vwap_premarket)
    orb30_vs_vwap_premarket = vs_level(orb30_close, vwap_premarket)
    ib_vs_vwap_premarket = vs_level(ib_close, vwap_premarket)

    orb15_vs_vwap_session = vs_level(orb15_close, vwap_session_at_orb15)
    orb30_vs_vwap_session = vs_level(orb30_close, vwap_session_at_orb30)
    ib_vs_vwap_session = vs_level(ib_close, vwap_session_at_ib)

    # ---- ORB meaningfulness vs. previous day's full-session range ----
    orb15_meaningful = (prev_range is not None and prev_range > 0 and
                        orb15_range >= prev_range * MIN_ORB_RANGE_FRACTION)
    orb30_meaningful = (prev_range is not None and prev_range > 0 and
                        orb30_range >= prev_range * MIN_ORB_RANGE_FRACTION)

    orb15_close_half_val = close_half(orb15_close, orb15_high, orb15_low) if orb15_meaningful else None
    orb15_direction = session_direction(orb15_open, orb15_close)
    orb15_mid = (orb15_high + orb15_low) / 2
    orb15_pos_in_ib = position_in_range(orb15_mid, ib_high, ib_low)
    orb15_pos_bucket = position_bucket(orb15_pos_in_ib)

    orb30_close_half_val = close_half(orb30_close, orb30_high, orb30_low) if orb30_meaningful else None
    orb30_direction = session_direction(orb30_open, orb30_close)
    orb30_mid = (orb30_high + orb30_low) / 2
    orb30_pos_in_ib = position_in_range(orb30_mid, ib_high, ib_low)
    orb30_pos_bucket = position_bucket(orb30_pos_in_ib)

    # ---- ORB-15 / ORB-30 internal extreme-first (live-safe: known the moment
    #      each ORB's own window closes, no look-ahead into later quarters) ----
    orb15_internal_extreme_first = which_extreme_first(orb15_data, orb15_high, orb15_low)
    orb30_internal_extreme_first = which_extreme_first(orb30_data, orb30_high, orb30_low)

    # ---- IB extreme first (uses the IB's FINAL high/low -- has look-ahead
    #      if the true extreme ends up set in Q3/Q4; useful for exploratory/
    #      diagnostic analysis, not as a signal you'd actually have at 10:00am) ----
    ib_extreme_first = which_extreme_first(ib_data, ib_high, ib_low)

    # ---- IB quarter / curve type ----
    ib_high_time = ib_data[ib_data['high'] == ib_high].index[0]
    ib_low_time = ib_data[ib_data['low'] == ib_low].index[0]
    ib_high_q = which_quarter(ib_high_time, q1_end, q2_end, q3_end)
    ib_low_q = which_quarter(ib_low_time, q1_end, q2_end, q3_end)
    ib_high_half = 1 if ib_high_time < q2_end else 2
    ib_low_half = 1 if ib_low_time < q2_end else 2
    if ib_high_half != ib_low_half:
        ib_curve_type = 'high_first' if ib_high_half == 1 else 'low_first'
    elif ib_high_half == 1:
        ib_curve_type = 'both_first_half'
    else:
        ib_curve_type = 'both_second_half'

    # ---- ORB-15 curve (breaking block = Q2, second half = Q3+Q4) ----
    orb15_curve = compute_curve_result(
        q2_data['high'].max(), q2_data['low'].min(),
        orb15_high, orb15_low,
        sh_after_orb15['high'].max(), sh_after_orb15['low'].min(),
    )

    # ---- ORB-30 curve, within-IB (breaking block = Q3, second half = Q4) ----
    orb30_curve_within_ib = compute_curve_result(
        q3_data['high'].max(), q3_data['low'].min(),
        orb30_high, orb30_low,
        q4_data['high'].max(), q4_data['low'].min(),
    )

    # ---- ORB-30 curve, post-IB (breaking block = Q3, second half = full post-IB) ----
    orb30_curve_post_ib = compute_curve_result(
        q3_data['high'].max(), q3_data['low'].min(),
        orb30_high, orb30_low,
        post_ib_data['high'].max(), post_ib_data['low'].min(),
    )

    # ---- Order flow: IB-level delta ----
    ib_volume = ib_data['volume'].sum()
    ib_delta = ib_data['delta'].sum()
    ib_delta_pct = ib_delta / ib_volume if ib_volume else np.nan
    ib_delta_bias = 'buy' if ib_delta > 0 else 'sell'
    ib_delta_divergence = (
        (ib_direction == 'bullish' and ib_delta_bias == 'sell') or
        (ib_direction == 'bearish' and ib_delta_bias == 'buy')
    )

    # ---- Order flow: delta POC + quartiles ----
    ib_cp = ib_data.copy()
    ib_cp['price_level'] = ((ib_cp['high'] + ib_cp['low']) / 2 / 0.25).round() * 0.25
    delta_profile = ib_cp.groupby('price_level')['delta'].sum()
    delta_poc_price = delta_profile.abs().idxmax()
    delta_poc_value = delta_profile.loc[delta_poc_price]
    delta_poc_pct = (delta_poc_price - ib_low) / ib_range
    delta_poc_zone = zone_label(delta_poc_pct)
    close_vs_poc = 'above' if ib_close > delta_poc_price else 'below'
    close_poc_distance = ib_close_pct - delta_poc_pct

    q_bounds = [
        ('q1', ib_low, ib_low + ib_range * 0.25),
        ('q2', ib_low + ib_range * 0.25, ib_low + ib_range * 0.50),
        ('q3', ib_low + ib_range * 0.50, ib_low + ib_range * 0.75),
        ('q4', ib_low + ib_range * 0.75, ib_high),
    ]
    q_deltas = {}
    for qname, qlo, qhi in q_bounds:
        m = (ib_cp['price_level'] >= qlo) & (ib_cp['price_level'] < qhi)
        q_deltas[f'delta_{qname}'] = ib_cp.loc[m, 'delta'].sum()

    delta_structure = delta_structure_label(
        q_deltas['delta_q1'], q_deltas['delta_q2'],
        q_deltas['delta_q3'], q_deltas['delta_q4'],
        ib_close_position,
    )

    # ---- Order flow: pre-IB windows ----
    pre_ib_stats = {}
    pre_ib_biases = []
    for win_name, (start_t, end_t) in PRE_IB_WINDOWS.items():
        win_bars = bars.loc[
            actual.replace(hour=int(start_t[:2]), minute=int(start_t[3:])):
            actual.replace(hour=int(end_t[:2]), minute=int(end_t[3:]))
        ]
        if win_bars.empty:
            pre_ib_stats[f'{win_name}_delta'] = np.nan
            pre_ib_stats[f'{win_name}_bias'] = None
            pre_ib_biases.append(None)
        else:
            w_delta = win_bars['delta'].sum()
            w_bias = 'buy' if w_delta > 0 else 'sell'
            pre_ib_stats[f'{win_name}_delta'] = w_delta
            pre_ib_stats[f'{win_name}_bias'] = w_bias
            pre_ib_biases.append(w_bias)

    # ---- Order flow: delta over the ORB windows themselves ----
    orb15_delta = orb15_data['delta'].sum()
    orb15_delta_bias = 'buy' if orb15_delta > 0 else 'sell'
    orb30_delta = orb30_data['delta'].sum()
    orb30_delta_bias = 'buy' if orb30_delta > 0 else 'sell'

    # ---- Chain agreement flags ----
    orb15_ib_close_agree = (orb15_close_half_val is not None and
                             orb15_close_half_val == ib_close_half_val)
    orb30_ib_close_agree = (orb30_close_half_val is not None and
                             orb30_close_half_val == ib_close_half_val)
    extreme_close_agree = None
    if ib_extreme_first in ('high', 'low'):
        expected = 'lower' if ib_extreme_first == 'high' else 'upper'
        extreme_close_agree = (ib_close_half_val == expected)

    # ---- Session context ----
    asia_open, asia_close = asia_data['open'].iloc[0], asia_data['close'].iloc[-1]
    asia_high, asia_low = asia_data['high'].max(), asia_data['low'].min()
    asia_dir = session_direction(asia_open, asia_close)

    lon_open, lon_close = lon_data['open'].iloc[0], lon_data['close'].iloc[-1]
    lon_high, lon_low = lon_data['high'].max(), lon_data['low'].min()
    lon_dir = session_direction(lon_open, lon_close)

    if pd.isna([asia_high, asia_low, lon_high, lon_low]).any():
        skipped += 1
        continue

    asia_london_pattern = classify_asia_london(lon_high, lon_low, asia_high, asia_low)

    lon_mid = (lon_high + lon_low) / 2
    overnight_high = max(asia_high, lon_high)
    overnight_low = min(asia_low, lon_low)
    overnight_mid = (overnight_high + overnight_low) / 2

    ny_8am_data = bars.loc[lon_end: lon_end + pd.Timedelta(minutes=1)]
    ny_8am_open = ny_8am_data['open'].iloc[0] if not ny_8am_data.empty else ib_open

    ny_open_vs_london_mid = 'above' if ny_8am_open > lon_mid else 'below'
    ny_open_vs_overnight_mid = 'above' if ny_8am_open > overnight_mid else 'below'

    # ---- NEW: position of ORB-15 / ORB-30 / IB relative to the London midpoint ----
    orb15_vs_london_mid = 'above' if orb15_close > lon_mid else 'below'
    orb30_vs_london_mid = 'above' if orb30_close > lon_mid else 'below'
    ib_vs_london_mid = 'above' if ib_close > lon_mid else 'below'

    # ---- Post-IB outcome / break tracking ----
    post_ib_close = post_ib_data['close'].iloc[-1]
    post_ib_dir = session_direction(ib_close, post_ib_close)

    one_hour_post_ib = bars.loc[ib_end:ib_end + pd.Timedelta(hours=1)]
    hb_1h, hb_t_1h, lb_1h, lb_t_1h = find_first_break(one_hour_post_ib, ib_high, ib_low)
    hb_ses, hb_t_ses, lb_ses, lb_t_ses = find_first_break(post_ib_data, ib_high, ib_low)

    first_break_dir = None
    for _, row in post_ib_data.iterrows():
        bh = row['high'] > ib_high
        bl = row['low'] < ib_low
        if bh and bl:
            first_break_dir = 'simultaneous'
            break
        elif bh:
            first_break_dir = 'high'
            break
        elif bl:
            first_break_dir = 'low'
            break

    trade_type = compute_trade_type(ib_extreme_first, ib_close_half_val)

    # ---- Structural score (10 signals: 2 each for ORB-15/ORB-30) ----
    struct_signals = [
        score_dir(asia_dir),
        score_dir(lon_dir),
        score_pattern(asia_london_pattern),
        score_close_half(orb15_close_half_val),
        score_position(orb15_pos_bucket),
        score_close_half(orb30_close_half_val),
        score_position(orb30_pos_bucket),
        score_extreme_first(ib_extreme_first),
        score_close_half(ib_close_half_val),
        score_ny_vs_mid(ny_open_vs_london_mid),
    ]
    struct_bias, struct_bull, struct_bear, struct_agreement = aggregate(
        struct_signals, MIN_STRUCTURAL_AGREEMENT)
    structural_score_label = struct_bias if struct_agreement else 'no_edge'

    # ---- Order-flow score (4 signals) ----
    of_signals = [
        score_delta_bias(ib_delta_bias),
        score_delta_structure(delta_structure),
        score_poc_alignment(close_vs_poc),
        score_pre_ib_majority(pre_ib_biases),
    ]
    of_bias, of_bull, of_bear, of_agreement = aggregate(of_signals, MIN_ORDERFLOW_AGREEMENT)
    orderflow_score_label = of_bias if of_agreement else 'no_edge'

    delta_confirms_structural = (
        structural_score_label in ('bullish', 'bearish') and
        orderflow_score_label == structural_score_label
    )

    record = {
        'date': actual.date(),
        'prev_range': prev_range,

        # session context
        'asia_direction': asia_dir, 'london_direction': lon_dir,
        'asia_london_pattern': asia_london_pattern,
        'ny_open_vs_london_mid': ny_open_vs_london_mid,
        'ny_open_vs_overnight_mid': ny_open_vs_overnight_mid,
        'lon_high': lon_high, 'lon_low': lon_low, 'lon_mid': lon_mid,
        'asia_high': asia_high, 'asia_low': asia_low,
        'overnight_high': overnight_high, 'overnight_low': overnight_low, 'overnight_mid': overnight_mid,

        # VWAP anchors
        'vwap_prevweek': vwap_prevweek, 'vwap_premarket': vwap_premarket,
        'vwap_weekly_at_orb15': vwap_weekly_at_orb15, 'vwap_weekly_at_orb30': vwap_weekly_at_orb30,
        'vwap_weekly_at_ib': vwap_weekly_at_ib,
        'vwap_session_at_orb15': vwap_session_at_orb15, 'vwap_session_at_orb30': vwap_session_at_orb30,
        'vwap_session_at_ib': vwap_session_at_ib,
        'orb15_vs_vwap_weekly': orb15_vs_vwap_weekly, 'orb30_vs_vwap_weekly': orb30_vs_vwap_weekly,
        'ib_vs_vwap_weekly': ib_vs_vwap_weekly,
        'orb15_vs_vwap_prevweek': orb15_vs_vwap_prevweek, 'orb30_vs_vwap_prevweek': orb30_vs_vwap_prevweek,
        'ib_vs_vwap_prevweek': ib_vs_vwap_prevweek,
        'orb15_vs_vwap_premarket': orb15_vs_vwap_premarket, 'orb30_vs_vwap_premarket': orb30_vs_vwap_premarket,
        'ib_vs_vwap_premarket': ib_vs_vwap_premarket,
        'orb15_vs_vwap_session': orb15_vs_vwap_session, 'orb30_vs_vwap_session': orb30_vs_vwap_session,
        'ib_vs_vwap_session': ib_vs_vwap_session,

        # 6-9am premarket session -- size, volume, delta, quartile profile shape
        'premarket_open': premarket_open, 'premarket_close': premarket_close,
        'premarket_high': premarket_high, 'premarket_low': premarket_low,
        'premarket_range': premarket_range, 'premarket_range_pct': premarket_range_pct,
        'premarket_direction': premarket_direction, 'premarket_close_half': premarket_close_half,
        'premarket_volume': premarket_volume, 'premarket_buy_vol': premarket_buy_vol,
        'premarket_sell_vol': premarket_sell_vol, 'premarket_delta': premarket_delta,
        'premarket_delta_bias': premarket_delta_bias,
        'premarket_profile_shape': premarket_profile['shape'] if premarket_profile else None,
        'premarket_v1_pct': premarket_profile['v1_pct'] if premarket_profile else np.nan,
        'premarket_v2_pct': premarket_profile['v2_pct'] if premarket_profile else np.nan,
        'premarket_v3_pct': premarket_profile['v3_pct'] if premarket_profile else np.nan,
        'premarket_v4_pct': premarket_profile['v4_pct'] if premarket_profile else np.nan,
        'premarket_d1': premarket_profile['d1'] if premarket_profile else np.nan,
        'premarket_d2': premarket_profile['d2'] if premarket_profile else np.nan,
        'premarket_d3': premarket_profile['d3'] if premarket_profile else np.nan,
        'premarket_d4': premarket_profile['d4'] if premarket_profile else np.nan,

        # ORB-15 (raw OHLC needed for Stage 3 positional analysis)
        'orb15_open': orb15_open, 'orb15_close': orb15_close,
        'orb15_high': orb15_high, 'orb15_low': orb15_low,
        'orb15_vs_london_mid': orb15_vs_london_mid,
        'orb15_volume': orb15_volume, 'orb15_buy_vol': orb15_buy_vol, 'orb15_sell_vol': orb15_sell_vol,
        'orb15_range_pct': orb15_range_pct,
        'orb15_profile_shape': orb15_profile['shape'] if orb15_profile else None,
        'orb15_v1_pct': orb15_profile['v1_pct'] if orb15_profile else np.nan,
        'orb15_v2_pct': orb15_profile['v2_pct'] if orb15_profile else np.nan,
        'orb15_v3_pct': orb15_profile['v3_pct'] if orb15_profile else np.nan,
        'orb15_v4_pct': orb15_profile['v4_pct'] if orb15_profile else np.nan,
        'orb15_d1': orb15_profile['d1'] if orb15_profile else np.nan,
        'orb15_d2': orb15_profile['d2'] if orb15_profile else np.nan,
        'orb15_d3': orb15_profile['d3'] if orb15_profile else np.nan,
        'orb15_d4': orb15_profile['d4'] if orb15_profile else np.nan,

        # ORB-15
        'orb15_range': orb15_range, 'orb15_meaningful': orb15_meaningful,
        'orb15_direction': orb15_direction, 'orb15_close_half': orb15_close_half_val,
        'orb15_pos_in_ib': orb15_pos_in_ib, 'orb15_pos_bucket': orb15_pos_bucket,
        'orb15_curve_result': orb15_curve,
        'orb15_ib_close_agree': orb15_ib_close_agree,
        'orb15_delta': orb15_delta, 'orb15_delta_bias': orb15_delta_bias,
        'orb15_internal_extreme_first': orb15_internal_extreme_first,

        # ORB-30
        'orb30_open': orb30_open, 'orb30_close': orb30_close,
        'orb30_high': orb30_high, 'orb30_low': orb30_low,
        'orb30_vs_london_mid': orb30_vs_london_mid,
        'orb30_volume': orb30_volume, 'orb30_buy_vol': orb30_buy_vol, 'orb30_sell_vol': orb30_sell_vol,
        'orb30_range_pct': orb30_range_pct,
        'orb30_profile_shape': orb30_profile['shape'] if orb30_profile else None,
        'orb30_v1_pct': orb30_profile['v1_pct'] if orb30_profile else np.nan,
        'orb30_v2_pct': orb30_profile['v2_pct'] if orb30_profile else np.nan,
        'orb30_v3_pct': orb30_profile['v3_pct'] if orb30_profile else np.nan,
        'orb30_v4_pct': orb30_profile['v4_pct'] if orb30_profile else np.nan,
        'orb30_d1': orb30_profile['d1'] if orb30_profile else np.nan,
        'orb30_d2': orb30_profile['d2'] if orb30_profile else np.nan,
        'orb30_d3': orb30_profile['d3'] if orb30_profile else np.nan,
        'orb30_d4': orb30_profile['d4'] if orb30_profile else np.nan,
        'orb30_range': orb30_range, 'orb30_meaningful': orb30_meaningful,
        'orb30_direction': orb30_direction, 'orb30_close_half': orb30_close_half_val,
        'orb30_pos_in_ib': orb30_pos_in_ib, 'orb30_pos_bucket': orb30_pos_bucket,
        'orb30_curve_result_within_ib': orb30_curve_within_ib,
        'orb30_curve_result_post_ib': orb30_curve_post_ib,
        'orb30_ib_close_agree': orb30_ib_close_agree,
        'orb30_delta': orb30_delta, 'orb30_delta_bias': orb30_delta_bias,
        'orb30_internal_extreme_first': orb30_internal_extreme_first,

        # IB structure
        'ib_open': ib_open, 'ib_close': ib_close, 'ib_high': ib_high, 'ib_low': ib_low,
        'ib_range': ib_range, 'ib_range_pct': ib_range_pct, 'ib_mid': ib_mid,
        'ib_direction': ib_direction, 'ib_close_position': ib_close_position,
        'ib_close_half': ib_close_half_val, 'ib_close_pct': ib_close_pct,
        'ib_close_zone': ib_close_zone,
        'ib_vs_london_mid': ib_vs_london_mid,
        'ib_extreme_first': ib_extreme_first,
        'ib_curve_type': ib_curve_type,
        'ib_high_quarter': ib_high_q, 'ib_low_quarter': ib_low_q,
        'extreme_close_agree': extreme_close_agree,

        # order flow -- IB level
        'ib_volume': ib_volume, 'ib_delta': ib_delta, 'ib_delta_pct': ib_delta_pct,
        'ib_buy_vol': ib_buy_vol, 'ib_sell_vol': ib_sell_vol,
        'ib_delta_bias': ib_delta_bias, 'ib_delta_divergence': ib_delta_divergence,
        'ib_profile_shape': ib_profile['shape'] if ib_profile else None,
        'ib_v1_pct': ib_profile['v1_pct'] if ib_profile else np.nan,
        'ib_v2_pct': ib_profile['v2_pct'] if ib_profile else np.nan,
        'ib_v3_pct': ib_profile['v3_pct'] if ib_profile else np.nan,
        'ib_v4_pct': ib_profile['v4_pct'] if ib_profile else np.nan,
        'delta_poc_price': delta_poc_price, 'delta_poc_value': delta_poc_value,
        'delta_poc_pct': delta_poc_pct, 'delta_poc_zone': delta_poc_zone,
        'close_vs_poc': close_vs_poc, 'close_poc_distance': close_poc_distance,
        **q_deltas,
        'delta_structure': delta_structure,
        **pre_ib_stats,

        # breaks / outcome
        'ib_high_broken_1h': hb_1h, 'ib_low_broken_1h': lb_1h,
        'ib_high_broken_session': hb_ses, 'ib_low_broken_session': lb_ses,
        'first_break_dir': first_break_dir,
        'post_ib_direction': post_ib_dir,

        # scoring
        'structural_bull_count': struct_bull, 'structural_bear_count': struct_bear,
        'structural_score_label': structural_score_label,
        'orderflow_bull_count': of_bull, 'orderflow_bear_count': of_bear,
        'orderflow_score_label': orderflow_score_label,
        'delta_confirms_structural': delta_confirms_structural,
        'trade_type': trade_type,
    }
    records.append(record)

    # carry this day's full-session range forward as next day's prev_range
    prev_range = full_day_data['high'].max() - full_day_data['low'].min()

rdf = pd.DataFrame(records)
print(f"\nBuilt {len(rdf):,} daily records | skipped {skipped}")

# ============================================================
# POST-LOOP: size & volume z-scores, percentiles, bands
# ============================================================
# Percentile rank and z-score need the full distribution, so these can only
# be computed after the day-by-day loop finishes, not inside it.

def zscore_band(z):
    if pd.isna(z):
        return None
    if z < -1.5:
        return 'Very Small'
    if z < -0.5:
        return 'Small'
    if z < 0.5:
        return 'Average'
    if z < 1.5:
        return 'Large'
    return 'Very Large'


SIZE_VOLUME_COLS = [
    'orb15_range_pct', 'orb30_range_pct', 'ib_range_pct', 'premarket_range_pct',
    'orb15_volume', 'orb30_volume', 'ib_volume', 'premarket_volume',
]
for col in SIZE_VOLUME_COLS:
    rdf[f'{col}_zscore'] = scipy_stats.zscore(rdf[col], nan_policy='omit')
    rdf[f'{col}_pctile'] = rdf[col].rank(pct=True) * 100
    rdf[f'{col}_band'] = rdf[f'{col}_zscore'].apply(zscore_band)

# buy-volume share (distinct from net delta -- volume can be huge and
# balanced, or small and lopsided; this captures the skew directly)
for prefix in ['orb15', 'orb30', 'ib', 'premarket']:
    buy = rdf[f'{prefix}_buy_vol']
    sell = rdf[f'{prefix}_sell_vol']
    total = buy + sell
    rdf[f'{prefix}_buy_vol_pct'] = np.where(total > 0, buy / total * 100, np.nan)

print("\nSize band distribution (IB range %):")
print(rdf['ib_range_pct_band'].value_counts())
print("\nProfile shape distribution (IB):")
print(rdf['ib_profile_shape'].value_counts())
print("\nProfile shape distribution (ORB-15):")
print(rdf['orb15_profile_shape'].value_counts())
print("\nProfile shape distribution (ORB-30):")
print(rdf['orb30_profile_shape'].value_counts())
print("\nProfile shape distribution (premarket 6-9am):")
print(rdf['premarket_profile_shape'].value_counts())

print("\nStructural score distribution:")
print(rdf['structural_score_label'].value_counts())
print("\nOrder-flow score distribution:")
print(rdf['orderflow_score_label'].value_counts())
print(f"\nDelta confirms structural score on {rdf['delta_confirms_structural'].sum():,} days")

rdf.to_csv(OUTPUT_DATASET_PATH, index=False)
print(f"\nSaved unified dataset -> {OUTPUT_DATASET_PATH}")
print("Stage 3 (analysis sections + ranked edge tables + the delta-lift test)")
print("will load this CSV directly.")

Loading combined 1-min bars...
  2,232,959 bars | 2020-01-01 18:00:00 -> 2026-04-29 23:59:00
  VWAP preprocessing done: 331 weeks, 1967 sessions
  1970 candidate trading days
  Day 250 / 1969 ...
  Day 500 / 1969 ...
  Day 750 / 1969 ...
  Day 1000 / 1969 ...
  Day 1250 / 1969 ...
  Day 1500 / 1969 ...
  Day 1750 / 1969 ...

Built 1,627 daily records | skipped 342

Size band distribution (IB range %):
Average       735
Small         534
Large         262
Very Large     95
Very Small      1
Name: ib_range_pct_band, dtype: int64

Profile shape distribution (IB):
D    566
P    545
b    405
B    111
Name: ib_profile_shape, dtype: int64

Profile shape distribution (ORB-15):
P    558
b    499
D    483
B     87
Name: orb15_profile_shape, dtype: int64

Profile shape distribution (ORB-30):
P    573
D    517
b    441
B     96
Name: orb30_profile_shape, dtype: int64

Structural score distribution:
no_edge    654
bullish    566
bearish    407
Name: structural_score_label, dtype: int64

Order-flow 

In [18]:
"""
NQ IB / ORB / Order-Flow Pipeline -- v5
Stage 3 (rebuilt): direct structural mapping of what influences IB formation,
then a delta confirmation layer on top of each finding.

Organized around six questions:
  1. If the ORB is bullish, is the IB bullish (and vice versa)?
  2. Break rates (1h/session, high/low) by IB close half and extreme-first.
  3. Where does the ORB sit in the IB, conditioned on which IB extreme formed
     first, and on which direction broke first post-IB?
  4. Where do the ORBs / IB sit relative to the London midpoint?
  5. How does that change across the four Asia/London structural patterns?
  6. Where does delta concentrate in the IB, and does it confirm (or add to)
     items 1-5?

Primary metric is break rate (H1h/L1h/Hses/Lses -- whether the IB high/low
gets taken out within 1h and within the session), since that's the
practical trading question. Post-IB direction (bull/bear close vs. IB close)
is shown alongside as a secondary outcome.
"""

import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CONFIG
# ============================================================

DATA_DIR = Path(r"C:\Users\tammm\NQ and ES analysis")
DATASET_PATH = DATA_DIR / "nq_v5_unified_dataset.csv"
RANKED_EDGE_PATH = DATA_DIR / "nq_v5_item_ranked_edge_table.csv"

MIN_DISPLAY_SAMPLE = 30
buckets = ['bottom_third', 'middle_third', 'top_third']

# ============================================================
# HELPERS
# ============================================================

def safe_pct(n, d):
    return (n / d * 100) if d > 0 else 0.0


def print_header(text, width=95):
    print("\n" + "=" * width)
    print(text.center(width))
    print("=" * width)


def print_subheader(text, width=95):
    print("\n" + "-" * width)
    print(f"  {text}")
    print("-" * width)


def print_row(label, count, total, indent=4):
    pct = safe_pct(count, total)
    bar = "#" * int(pct / 2)
    print(f"{' '*indent}{label:<45} {count:5d}  ({pct:5.1f}%)  {bar}")


def outcome_stats(subset, direction_col='post_ib_direction', min_n=MIN_DISPLAY_SAMPLE):
    bull = (subset[direction_col] == 'bullish').sum()
    bear = (subset[direction_col] == 'bearish').sum()
    n = bull + bear
    if n < min_n:
        return None
    bull_pct = safe_pct(bull, n)
    return {'n': n, 'bull_pct': bull_pct, 'bear_pct': 100 - bull_pct,
            'edge_vs_50': abs(bull_pct - 50), 'bias': 'BULL' if bull_pct > 50 else 'BEAR'}


def print_outcome(label, subset, indent=4, min_n=MIN_DISPLAY_SAMPLE, direction_col='post_ib_direction'):
    s = outcome_stats(subset, direction_col=direction_col, min_n=min_n)
    if s is None:
        print(f"{' '*indent}{label:<45} n={len(subset):4d}  (below min sample)")
        return
    print(f"{' '*indent}{label:<45} n={s['n']:4d}  "
          f"Bull:{s['bull_pct']:5.1f}%  Bear:{s['bear_pct']:5.1f}%  Edge:{s['edge_vs_50']:4.1f}%  [{s['bias']}]")


def break_stats(subset, min_n=MIN_DISPLAY_SAMPLE):
    n = len(subset)
    if n < min_n:
        return None
    hb1 = subset['ib_high_broken_1h'].sum()
    lb1 = subset['ib_low_broken_1h'].sum()
    hbs = subset['ib_high_broken_session'].sum()
    lbs = subset['ib_low_broken_session'].sum()
    return {'n': n, 'h1h': safe_pct(hb1, n), 'l1h': safe_pct(lb1, n),
            'hses': safe_pct(hbs, n), 'lses': safe_pct(lbs, n)}


def print_break(label, subset, indent=4, min_n=MIN_DISPLAY_SAMPLE):
    s = break_stats(subset, min_n=min_n)
    if s is None:
        print(f"{' '*indent}{label:<45} n={len(subset):4d}  (below min sample)")
        return
    print(f"{' '*indent}{label:<45} n={s['n']:4d}  "
          f"H1h={s['h1h']:5.1f}%  L1h={s['l1h']:5.1f}%  Hses={s['hses']:5.1f}%  Lses={s['lses']:5.1f}%")


def direction_concordance(df, a_col, b_col, a_name, b_name, min_n=MIN_DISPLAY_SAMPLE):
    sub = df[df[a_col].notna() & df[b_col].notna()]

    # Raw joint counts first, so the two conditional views below are
    # visibly just two slices of the same table (Bayes' theorem, not a
    # discrepancy): forward = row %, "vice versa" = column %.
    bb = ((sub[a_col] == 'bullish') & (sub[b_col] == 'bullish')).sum()
    bear_b = ((sub[a_col] == 'bullish') & (sub[b_col] == 'bearish')).sum()
    b_bear = ((sub[a_col] == 'bearish') & (sub[b_col] == 'bullish')).sum()
    bearbear = ((sub[a_col] == 'bearish') & (sub[b_col] == 'bearish')).sum()
    print(f"\n  Raw joint counts (n={len(sub)}):")
    print(f"  {'':<18}{b_name+' bull':>12}{b_name+' bear':>12}{'Total':>10}")
    print(f"  {a_name+' bull':<18}{bb:>12}{bear_b:>12}{bb+bear_b:>10}")
    print(f"  {a_name+' bear':<18}{b_bear:>12}{bearbear:>12}{b_bear+bearbear:>10}")
    print(f"  {'Total':<18}{bb+b_bear:>12}{bear_b+bearbear:>12}{len(sub):>10}")

    print(f"\n  {a_name} direction -> {b_name} direction  (row %, the tradeable direction)")
    for a_val in ['bullish', 'bearish']:
        s = sub[sub[a_col] == a_val]
        n = len(s)
        if n < min_n:
            print(f"    {a_name}={a_val:<8} (n={n:4d})  below min sample")
            continue
        bull = (s[b_col] == 'bullish').sum()
        bear = (s[b_col] == 'bearish').sum()
        print(f"    {a_name}={a_val:<8} (n={n:4d})  -> {b_name} bullish {safe_pct(bull,n):5.1f}%  "
              f"{b_name} bearish {safe_pct(bear,n):5.1f}%")
    print(f"\n  {b_name} direction -> {a_name} direction  (column %, 'the vice versa' -- diagnostic, not tradeable)")
    for b_val in ['bullish', 'bearish']:
        s = sub[sub[b_col] == b_val]
        n = len(s)
        if n < min_n:
            print(f"    {b_name}={b_val:<8} (n={n:4d})  below min sample")
            continue
        bull = (s[a_col] == 'bullish').sum()
        bear = (s[a_col] == 'bearish').sum()
        print(f"    {b_name}={b_val:<8} (n={n:4d})  -> {a_name} bullish {safe_pct(bull,n):5.1f}%  "
              f"{a_name} bearish {safe_pct(bear,n):5.1f}%")


def print_position_conditional(cond_col, cond_df, pos_col, base_rates):
    for cv in ['high', 'low']:
        subset = cond_df[cond_df[cond_col] == cv]
        n = len(subset)
        print(f"\n  {cond_col}={cv} (n={n}) -> {pos_col}")
        for bucket in buckets:
            count = (subset[pos_col] == bucket).sum()
            pct = safe_pct(count, n)
            b = base_rates[bucket]
            lift = round(pct / b, 2) if b > 0 else 0
            print(f"    {bucket:<15} {count:4d}  {pct:5.1f}%  (base {b:5.1f}%, lift {lift:.2f}x)")


edge_rows = []


def collect_edge(label, subset, base_bull_pct, min_n=MIN_DISPLAY_SAMPLE):
    s = outcome_stats(subset, min_n=min_n)
    if s is None:
        return
    edge_rows.append({'label': label, 'n': s['n'], 'bull_pct': s['bull_pct'],
                       'bear_pct': s['bear_pct'], 'edge_vs_base': abs(s['bull_pct'] - base_bull_pct),
                       'bias': s['bias']})


# ============================================================
# LOAD
# ============================================================

print("Loading unified dataset...")
rdf = pd.read_csv(DATASET_PATH, parse_dates=['date'])
total_days = len(rdf)
print(f"  {total_days:,} days loaded")

REQUIRED_COLUMNS = [
    'post_ib_direction', 'ib_direction', 'orb15_direction', 'orb30_direction',
    'ib_close_half', 'ib_extreme_first', 'orb15_pos_bucket', 'orb30_pos_bucket',
    'orb15_close_half', 'orb30_close_half', 'orb15_meaningful', 'orb30_meaningful',
    'orb15_internal_extreme_first', 'orb30_internal_extreme_first',
    'orb15_vs_london_mid', 'orb30_vs_london_mid', 'ib_vs_london_mid',
    'asia_london_pattern', 'ib_delta_bias', 'delta_poc_zone', 'close_vs_poc',
    'delta_structure', 'first_break_dir', 'ib_high_broken_1h', 'ib_low_broken_1h',
    'ib_high_broken_session', 'ib_low_broken_session',
]
missing = [c for c in REQUIRED_COLUMNS if c not in rdf.columns]
if missing:
    raise SystemExit(
        "\n\nSTOPPING -- the loaded CSV is missing columns this script needs:\n"
        f"  {missing}\n\n"
        "nq_v5_unified_dataset.csv was built by an OLDER version of Stage 2 than\n"
        "the one currently saved. Rerun nq_v5_stage2_daily_features.py to regenerate\n"
        "the CSV with the current schema, then rerun this script.\n"
    )

BASE_STATS = outcome_stats(rdf, min_n=1)
BASE_BULL_PCT = BASE_STATS['bull_pct'] if BASE_STATS else 50.0
print(f"  Baseline post-IB direction: Bull {BASE_BULL_PCT:.1f}%  (n={BASE_STATS['n'] if BASE_STATS else 0})")

ef = rdf[rdf['ib_extreme_first'].isin(['high', 'low'])]
fb = rdf[rdf['first_break_dir'].isin(['high', 'low'])]
meaningful15 = rdf[rdf['orb15_meaningful'] == True]
meaningful30 = rdf[rdf['orb30_meaningful'] == True]


# ============================================================
# ITEM 1 -- ORB DIRECTION <-> IB DIRECTION
# ============================================================

print_header("ITEM 1 -- IF THE ORB IS BULLISH, IS THE IB BULLISH? (AND VICE VERSA)")

direction_concordance(rdf, 'orb15_direction', 'ib_direction', 'ORB-15', 'IB')
direction_concordance(rdf, 'orb30_direction', 'ib_direction', 'ORB-30', 'IB')


# ============================================================
# ITEM 2 -- BREAK RATES BY IB CLOSE HALF AND EXTREME-FIRST
# ============================================================

print_header("ITEM 2 -- BREAK RATES BY IB CLOSE HALF AND EXTREME-FIRST")

print_subheader("Break rates by IB close half (marginal)")
for ch in ['upper', 'lower']:
    print_break(f"ib_close={ch}", rdf[rdf['ib_close_half'] == ch])

print_subheader("Break rates by IB extreme-first (marginal)")
for ext in ['high', 'low']:
    print_break(f"extreme_first={ext}", rdf[rdf['ib_extreme_first'] == ext])

print_subheader("Break rates by IB close half x extreme-first (joint, 2x2)")
for ch in ['upper', 'lower']:
    for ext in ['high', 'low']:
        subset = rdf[(rdf['ib_close_half'] == ch) & (rdf['ib_extreme_first'] == ext)]
        print_break(f"ib_close={ch}+extreme={ext}", subset)
        collect_edge(f"ib_close={ch}+extreme={ext}", subset, BASE_BULL_PCT)


# ============================================================
# ITEM 3 -- ORB POSITION IN IB, CONDITIONED ON EXTREME-FIRST / BREAK-FIRST
# ============================================================

print_header("ITEM 3 -- ORB POSITION IN IB, CONDITIONED ON EXTREME-FIRST AND BREAK-FIRST")
print("  (Note: this is the ORBs' position within the IB, not the IB's own structure.)\n")

base_ef_b15 = {b: safe_pct((ef['orb15_pos_bucket'] == b).sum(), len(ef)) for b in buckets}
base_ef_b30 = {b: safe_pct((ef['orb30_pos_bucket'] == b).sum(), len(ef)) for b in buckets}
base_fb_b15 = {b: safe_pct((fb['orb15_pos_bucket'] == b).sum(), len(fb)) for b in buckets}
base_fb_b30 = {b: safe_pct((fb['orb30_pos_bucket'] == b).sum(), len(fb)) for b in buckets}

print_subheader(f"ORB-15 position, conditioned on which IB extreme formed FIRST (n={len(ef)})")
print_position_conditional('ib_extreme_first', ef, 'orb15_pos_bucket', base_ef_b15)

print_subheader(f"ORB-30 position, conditioned on which IB extreme formed FIRST (n={len(ef)})")
print_position_conditional('ib_extreme_first', ef, 'orb30_pos_bucket', base_ef_b30)

print_subheader(f"ORB-15 position, conditioned on which direction BROKE FIRST post-IB (n={len(fb)})")
print_position_conditional('first_break_dir', fb, 'orb15_pos_bucket', base_fb_b15)

print_subheader(f"ORB-30 position, conditioned on which direction BROKE FIRST post-IB (n={len(fb)})")
print_position_conditional('first_break_dir', fb, 'orb30_pos_bucket', base_fb_b30)


# ============================================================
# TRADING FRAMEWORK -- 10:00-10:30 DECISION WINDOW
# ============================================================
# Given what's known at the ORB-30 close (10:00am) plus which extreme has
# formed first, what % of the time does the IB itself (9:30-10:30) end up
# bullish vs. bearish? Two versions of "extreme first" are shown:
#
#   ib_extreme_first          -- uses the IB's FINAL high/low. If the true
#                                 extreme ends up printing in Q3/Q4 (after
#                                 10:00am), this has look-ahead baked in --
#                                 exploratory/diagnostic only, not something
#                                 you actually know at 10:00am.
#   orb30_internal_extreme_first -- whether ORB-30's OWN high or low (within
#                                 its own 9:30-10:00 window) printed first.
#                                 Fully known the moment ORB-30 closes --
#                                 this is the live-safe version.
#
# Comparing the two tells you how much the look-ahead in ib_extreme_first
# is actually inflating the apparent edge.
# ============================================================

print_header("TRADING FRAMEWORK -- 10:00-10:30 DECISION WINDOW")
print("  Given ORB close-half + which extreme formed first, what % of the time")
print("  does the IB itself end up bullish vs. bearish?\n")
print("  ib_extreme_first uses the IB's FINAL high/low -- has look-ahead if the")
print("  true extreme prints in Q3/Q4. orb30_internal_extreme_first is fully known")
print("  at 10:00am (ORB-30's own high/low order) -- this is the live-safe version.\n")

for label, mdf, ch_col in [('ORB-15', meaningful15, 'orb15_close_half'),
                           ('ORB-30', meaningful30, 'orb30_close_half')]:
    print_subheader(f"{label} close-half x IB extreme-first (LOOK-AHEAD) -> IB direction")
    for ch in ['upper', 'lower']:
        for ext in ['high', 'low']:
            subset = mdf[(mdf[ch_col] == ch) & (mdf['ib_extreme_first'] == ext)]
            print_outcome(f"{label}={ch}+ib_extreme_first={ext}", subset, direction_col='ib_direction')
            collect_edge(f"{label}={ch}+ib_extreme_first={ext}", subset, BASE_BULL_PCT)

print_subheader("ORB-30 close-half x ORB-30's OWN extreme-first (LIVE-SAFE) -> IB direction")
for ch in ['upper', 'lower']:
    for ext in ['high', 'low']:
        subset = meaningful30[(meaningful30['orb30_close_half'] == ch) &
                              (meaningful30['orb30_internal_extreme_first'] == ext)]
        print_outcome(f"ORB-30={ch}+orb30_internal_extreme_first={ext}", subset, direction_col='ib_direction')
        collect_edge(f"ORB-30={ch}+orb30_internal_extreme_first={ext}", subset, BASE_BULL_PCT)

print_subheader("ORB-15 close-half x ORB-15's OWN extreme-first (LIVE-SAFE at 9:45am) -> IB direction")
for ch in ['upper', 'lower']:
    for ext in ['high', 'low']:
        subset = meaningful15[(meaningful15['orb15_close_half'] == ch) &
                              (meaningful15['orb15_internal_extreme_first'] == ext)]
        print_outcome(f"ORB-15={ch}+orb15_internal_extreme_first={ext}", subset, direction_col='ib_direction')
        collect_edge(f"ORB-15={ch}+orb15_internal_extreme_first={ext}", subset, BASE_BULL_PCT)


# ============================================================
# ITEM 4 -- ORB / IB POSITION RELATIVE TO LONDON MIDPOINT
# ============================================================

print_header("ITEM 4 -- POSITION OF ORBS / IB RELATIVE TO THE LONDON MIDPOINT")

for label, col in [('ORB-15', 'orb15_vs_london_mid'), ('ORB-30', 'orb30_vs_london_mid'), ('IB', 'ib_vs_london_mid')]:
    print_subheader(f"{label} close vs. London midpoint")
    for pos in ['above', 'below']:
        print_row(f"{label} closes {pos} London mid", (rdf[col] == pos).sum(), total_days)
    print()
    for pos in ['above', 'below']:
        subset = rdf[rdf[col] == pos]
        print_outcome(f"{label} {pos} London mid -> Post-IB", subset)
        print_break(f"{label} {pos} London mid", subset)
        collect_edge(f"{label}_vs_london_mid={pos}", subset, BASE_BULL_PCT)


# ============================================================
# ITEM 5 -- ORB / IB POSITION BY SESSION STRUCTURAL PATTERN
# ============================================================

print_header("ITEM 5 -- ORB/IB POSITION BY ASIA/LONDON STRUCTURAL PATTERN")

for pat in ['london_engulfs_asia', 'asia_engulfs_london',
            'london_partially_engulfs_upwards', 'london_partially_engulfs_downwards']:
    pat_sub = rdf[rdf['asia_london_pattern'] == pat]
    n_pat = len(pat_sub)
    print_subheader(f"{pat}  (n={n_pat})")
    if n_pat < MIN_DISPLAY_SAMPLE:
        print("    below min sample")
        continue
    for label, col in [('ORB-15', 'orb15_vs_london_mid'), ('ORB-30', 'orb30_vs_london_mid'), ('IB', 'ib_vs_london_mid')]:
        above = (pat_sub[col] == 'above').sum()
        print(f"    {label} closes above London mid: {safe_pct(above, n_pat):5.1f}%  ({above}/{n_pat})")
    print()
    print_outcome(f"{pat} -> Post-IB", pat_sub)
    print_break(pat, pat_sub)
    collect_edge(f"asia_london_pattern={pat}", pat_sub, BASE_BULL_PCT)

    for label, col in [('ORB-15', 'orb15_vs_london_mid'), ('IB', 'ib_vs_london_mid')]:
        for pos in ['above', 'below']:
            sub2 = pat_sub[pat_sub[col] == pos]
            print_outcome(f"  {pat[:24]} + {label}={pos}", sub2)
            collect_edge(f"pattern={pat[:24]}+{label}_vs_lonmid={pos}", sub2, BASE_BULL_PCT)


# ============================================================
# ITEM 6 -- ORDER FLOW: WHERE DELTA CONCENTRATES, AND DOES IT CONFIRM 1-5?
# ============================================================

print_header("ITEM 6 -- ORDER FLOW: WHERE DELTA CONCENTRATES, AND DOES IT CONFIRM ITEMS 1-5?")

print_subheader("Net IB delta bias -- distribution + outcomes")
for bias in ['buy', 'sell']:
    subset = rdf[rdf['ib_delta_bias'] == bias]
    print_row(f"ib_delta_bias={bias}", len(subset), total_days)
for bias in ['buy', 'sell']:
    subset = rdf[rdf['ib_delta_bias'] == bias]
    print_outcome(f"ib_delta_bias={bias}", subset)
    print_break(f"ib_delta_bias={bias}", subset)
    collect_edge(f"ib_delta_bias={bias}", subset, BASE_BULL_PCT)

print_subheader("Delta POC zone -- where within the IB range does delta concentrate")
print("  (Q1/Q2 = bottom half of IB, Q3/Q4 = top half)\n")
for zone in ['Q1', 'Q2', 'Q3', 'Q4']:
    print_row(f"delta_poc_zone={zone}", (rdf['delta_poc_zone'] == zone).sum(), total_days)
print()
for zone in ['Q1', 'Q2', 'Q3', 'Q4']:
    subset = rdf[rdf['delta_poc_zone'] == zone]
    print_outcome(f"delta_poc_zone={zone}", subset)
    collect_edge(f"delta_poc_zone={zone}", subset, BASE_BULL_PCT)

print_subheader("Close vs. delta POC -- does price close above or below where delta concentrated?")
for val in ['above', 'below']:
    subset = rdf[rdf['close_vs_poc'] == val]
    print_outcome(f"close_vs_poc={val}", subset)
    print_break(f"close_vs_poc={val}", subset)
    collect_edge(f"close_vs_poc={val}", subset, BASE_BULL_PCT)

print_subheader("Delta-by-quartile structure label -- distribution + outcomes")
for label in ['ideal_bullish', 'buying_only', 'absorbed_no_confirm', 'weak_bullish',
              'ideal_bearish', 'selling_only', 'weak_bearish']:
    subset = rdf[rdf['delta_structure'] == label]
    n = len(subset)
    if n == 0:
        continue
    print_outcome(label, subset)
    print_break(label, subset)
    collect_edge(f"delta_structure={label}", subset, BASE_BULL_PCT)

# ---- Delta layered onto item 1 (direction concordance) ----
print_subheader("Item 1 + delta: does IB delta bias confirm ORB direction?")
for orb_label, orb_col in [('ORB-15', 'orb15_direction'), ('ORB-30', 'orb30_direction')]:
    for d in ['bullish', 'bearish']:
        s = rdf[rdf[orb_col] == d]
        n = len(s)
        if n < MIN_DISPLAY_SAMPLE:
            continue
        expected_bias = 'buy' if d == 'bullish' else 'sell'
        agree = (s['ib_delta_bias'] == expected_bias).sum()
        print(f"    {orb_label}={d:<8} (n={n:4d})  IB delta agrees with ORB direction: {safe_pct(agree,n):5.1f}%")

# ---- Delta layered onto item 2 (break rates by close-half x extreme-first) ----
print_subheader("Item 2 + delta: break rates by close-half x extreme-first x delta bias")
for ch in ['upper', 'lower']:
    for ext in ['high', 'low']:
        for bias in ['buy', 'sell']:
            subset = rdf[(rdf['ib_close_half'] == ch) & (rdf['ib_extreme_first'] == ext) &
                         (rdf['ib_delta_bias'] == bias)]
            print_break(f"ib={ch}+ext={ext}+delta={bias}", subset)
            collect_edge(f"ib={ch}+ext={ext}+delta={bias}", subset, BASE_BULL_PCT)

# ---- Delta layered onto item 3 (ORB position vs. delta POC zone) ----
print_subheader("Item 3 + delta: does delta POC zone sit on the same side of the IB as the ORB?")
for bucket, poc_zones in [('bottom_third', ['Q1', 'Q2']), ('top_third', ['Q3', 'Q4'])]:
    subset = rdf[(rdf['orb15_pos_bucket'] == bucket) & (rdf['delta_poc_zone'].isin(poc_zones))]
    label = f"orb15={bucket}+poc_zone in {poc_zones}"
    print_outcome(label, subset)
    print_break(label, subset)
    collect_edge(label, subset, BASE_BULL_PCT)

# ---- Delta layered onto items 4/5 (position vs. London mid x delta bias) ----
print_subheader("Item 4 + delta: position vs. London mid x IB delta bias")
for label, col in [('ORB-15', 'orb15_vs_london_mid'), ('IB', 'ib_vs_london_mid')]:
    for pos in ['above', 'below']:
        for bias in ['buy', 'sell']:
            subset = rdf[(rdf[col] == pos) & (rdf['ib_delta_bias'] == bias)]
            key = f"{label}_vs_lonmid={pos}+delta={bias}"
            print_outcome(key, subset)
            collect_edge(key, subset, BASE_BULL_PCT)


# ============================================================
# RANKED EDGE TABLE -- everything collected across items 1-6
# ============================================================

print_header("RANKED EDGE TABLE -- ALL PATTERNS FROM ITEMS 1-6, SORTED BY EDGE")
print(f"  Baseline Bull%: {BASE_BULL_PCT:.1f}  |  min n={MIN_DISPLAY_SAMPLE}\n")

edge_df = pd.DataFrame(edge_rows).drop_duplicates(subset=['label']).sort_values('edge_vs_base', ascending=False)
print(f"  {'PATTERN':<55}{'n':>6}{'BULL%':>8}{'BEAR%':>8}{'EDGE':>8}{'BIAS':>6}")
for _, row in edge_df.iterrows():
    print(f"  {row['label']:<55}{row['n']:>6}{row['bull_pct']:>7.1f}%{row['bear_pct']:>7.1f}%"
          f"{row['edge_vs_base']:>7.1f}%{row['bias']:>6}")

edge_df.to_csv(RANKED_EDGE_PATH, index=False)
print(f"\nSaved -> {RANKED_EDGE_PATH}")

print_header("ANALYSIS COMPLETE")

Loading unified dataset...
  1,627 days loaded
  Baseline post-IB direction: Bull 55.7%  (n=1627)

              ITEM 1 -- IF THE ORB IS BULLISH, IS THE IB BULLISH? (AND VICE VERSA)             

  Raw joint counts (n=1626):
                         IB bull     IB bear     Total
  ORB-15 bull                623         254       877
  ORB-15 bear                223         526       749
  Total                      846         780      1626

  ORB-15 direction -> IB direction  (row %, the tradeable direction)
    ORB-15=bullish  (n= 877)  -> IB bullish  71.0%  IB bearish  29.0%
    ORB-15=bearish  (n= 749)  -> IB bullish  29.8%  IB bearish  70.2%

  IB direction -> ORB-15 direction  (column %, 'the vice versa' -- diagnostic, not tradeable)
    IB=bullish  (n= 846)  -> ORB-15 bullish  73.6%  ORB-15 bearish  26.4%
    IB=bearish  (n= 780)  -> ORB-15 bullish  32.6%  ORB-15 bearish  67.4%

  Raw joint counts (n=1626):
                         IB bull     IB bear     Total
  ORB-30 bull    

    ORB-30 below London mid -> Post-IB            n= 751  Bull: 51.7%  Bear: 48.3%  Edge: 1.7%  [BULL]
    ORB-30 below London mid                       n= 751  H1h= 30.4%  L1h= 43.4%  Hses= 52.9%  Lses= 64.8%

-----------------------------------------------------------------------------------------------
  IB close vs. London midpoint
-----------------------------------------------------------------------------------------------
    IB closes above London mid                      845  ( 51.9%)  #########################
    IB closes below London mid                      782  ( 48.1%)  ########################

    IB above London mid -> Post-IB                n= 845  Bull: 59.3%  Bear: 40.7%  Edge: 9.3%  [BULL]
    IB above London mid                           n= 845  H1h= 59.1%  L1h= 20.5%  Hses= 77.8%  Lses= 41.2%
    IB below London mid -> Post-IB                n= 782  Bull: 51.8%  Bear: 48.2%  Edge: 1.8%  [BULL]
    IB below London mid                           n= 782  H1h= 21.7

    buying_only                                   n= 408  Bull: 59.6%  Bear: 40.4%  Edge: 9.6%  [BULL]
    buying_only                                   n= 408  H1h= 65.0%  L1h= 12.0%  Hses= 82.6%  Lses= 32.8%
    absorbed_no_confirm                           n= 108  Bull: 54.6%  Bear: 45.4%  Edge: 4.6%  [BULL]
    absorbed_no_confirm                           n= 108  H1h= 30.6%  L1h= 34.3%  Hses= 59.3%  Lses= 58.3%
    weak_bullish                                  n=  61  Bull: 63.9%  Bear: 36.1%  Edge:13.9%  [BULL]
    weak_bullish                                  n=  61  H1h= 59.0%  L1h= 11.5%  Hses= 77.0%  Lses= 26.2%
    ideal_bearish                                 n= 279  Bull: 49.8%  Bear: 50.2%  Edge: 0.2%  [BEAR]
    ideal_bearish                                 n= 279  H1h= 11.5%  L1h= 63.1%  Hses= 39.4%  Lses= 82.4%
    selling_only                                  n= 357  Bull: 51.3%  Bear: 48.7%  Edge: 1.3%  [BULL]
    selling_only                                  n= 357 

In [28]:
"""
NQ IB / ORB / Order-Flow Pipeline -- v5
Stage 4: size, volume, and delta/volume profile shape (P/b/B/D).

Three parts:
  A. SIZE     -- is the ORB/IB range larger or smaller than normal
                 (z-score bands off the full 2020-2026 distribution)?
  B. VOLUME   -- is volume higher/lower than normal, and is it mostly
                 buy- or sell-initiated (buy_vol_pct, distinct from net delta)?
  C. SHAPE    -- P/b/B/D volume-by-quartile shape for ORB-15/ORB-30/IB, then
                 paired with the delta sign in the dominant quartile(s) to
                 separate continuation-confirming days from exhaustion/
                 absorption warnings, and (for bimodal B-shape) whether the
                 two volume nodes agree or fight.

Requires the Stage 2 CSV to include the size/volume/profile fields added in
the latest patch -- rerun Stage 2 first if the schema check below fails.
"""

import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CONFIG
# ============================================================

DATA_DIR = Path(r"C:\Users\tammm\NQ and ES analysis")
DATASET_PATH = DATA_DIR / "nq_v5_unified_dataset.csv"
RANKED_EDGE_PATH = DATA_DIR / "nq_v5_size_volume_profile_edge_table.csv"

MIN_DISPLAY_SAMPLE = 30
BAND_ORDER = ['Very Small', 'Small', 'Average', 'Large', 'Very Large']
SHAPE_ORDER = ['P', 'b', 'D', 'B']

# ============================================================
# HELPERS (same conventions as Stage 3)
# ============================================================

def safe_pct(n, d):
    return (n / d * 100) if d > 0 else 0.0


def print_header(text, width=95):
    print("\n" + "=" * width)
    print(text.center(width))
    print("=" * width)


def print_subheader(text, width=95):
    print("\n" + "-" * width)
    print(f"  {text}")
    print("-" * width)


def print_row(label, count, total, indent=4):
    pct = safe_pct(count, total)
    bar = "#" * int(pct / 2)
    print(f"{' '*indent}{label:<45} {count:5d}  ({pct:5.1f}%)  {bar}")


def outcome_stats(subset, direction_col='post_ib_direction', min_n=MIN_DISPLAY_SAMPLE):
    bull = (subset[direction_col] == 'bullish').sum()
    bear = (subset[direction_col] == 'bearish').sum()
    n = bull + bear
    if n < min_n:
        return None
    bull_pct = safe_pct(bull, n)
    return {'n': n, 'bull_pct': bull_pct, 'bear_pct': 100 - bull_pct,
            'edge_vs_50': abs(bull_pct - 50), 'bias': 'BULL' if bull_pct > 50 else 'BEAR'}


def print_outcome(label, subset, indent=4, min_n=MIN_DISPLAY_SAMPLE, direction_col='post_ib_direction'):
    s = outcome_stats(subset, direction_col=direction_col, min_n=min_n)
    if s is None:
        print(f"{' '*indent}{label:<45} n={len(subset):4d}  (below min sample)")
        return
    print(f"{' '*indent}{label:<45} n={s['n']:4d}  "
          f"Bull:{s['bull_pct']:5.1f}%  Bear:{s['bear_pct']:5.1f}%  Edge:{s['edge_vs_50']:4.1f}%  [{s['bias']}]")


def break_stats(subset, min_n=MIN_DISPLAY_SAMPLE):
    n = len(subset)
    if n < min_n:
        return None
    hb1 = subset['ib_high_broken_1h'].sum()
    lb1 = subset['ib_low_broken_1h'].sum()
    hbs = subset['ib_high_broken_session'].sum()
    lbs = subset['ib_low_broken_session'].sum()
    return {'n': n, 'h1h': safe_pct(hb1, n), 'l1h': safe_pct(lb1, n),
            'hses': safe_pct(hbs, n), 'lses': safe_pct(lbs, n)}


def print_break(label, subset, indent=4, min_n=MIN_DISPLAY_SAMPLE):
    s = break_stats(subset, min_n=min_n)
    if s is None:
        print(f"{' '*indent}{label:<45} n={len(subset):4d}  (below min sample)")
        return
    print(f"{' '*indent}{label:<45} n={s['n']:4d}  "
          f"H1h={s['h1h']:5.1f}%  L1h={s['l1h']:5.1f}%  Hses={s['hses']:5.1f}%  Lses={s['lses']:5.1f}%")


edge_rows = []


def collect_edge(label, subset, base_bull_pct, min_n=MIN_DISPLAY_SAMPLE, direction_col='post_ib_direction'):
    s = outcome_stats(subset, direction_col=direction_col, min_n=min_n)
    if s is None:
        return
    edge_rows.append({'label': label, 'n': s['n'], 'bull_pct': s['bull_pct'],
                       'bear_pct': s['bear_pct'], 'edge_vs_base': abs(s['bull_pct'] - base_bull_pct),
                       'bias': s['bias']})


def buy_vol_band(pct):
    """Superseded by recalibrated_buy_vol_band() below, which uses
    per-column empirical quantiles instead of these fixed thresholds --
    kept only as a comment for context, not called anywhere."""
    return None


# ============================================================
# LOAD
# ============================================================

print("Loading unified dataset...")
rdf = pd.read_csv(DATASET_PATH, parse_dates=['date'])
total_days = len(rdf)
print(f"  {total_days:,} days loaded")

REQUIRED_COLUMNS = [
    'orb15_range_pct_band', 'orb30_range_pct_band', 'ib_range_pct_band',
    'orb15_volume_band', 'orb30_volume_band', 'ib_volume_band',
    'orb15_buy_vol_pct', 'orb30_buy_vol_pct', 'ib_buy_vol_pct',
    'orb15_profile_shape', 'orb30_profile_shape', 'ib_profile_shape',
    'orb15_d1', 'orb15_d2', 'orb15_d3', 'orb15_d4',
    'orb30_d1', 'orb30_d2', 'orb30_d3', 'orb30_d4',
    'delta_q1', 'delta_q2', 'delta_q3', 'delta_q4',
    'ib_close_half', 'ib_extreme_first', 'ib_direction',
    'orb30_close_half', 'orb30_internal_extreme_first',
    'ib_high_broken_1h', 'ib_low_broken_1h',
    'ib_high_broken_session', 'ib_low_broken_session', 'post_ib_direction',
]
missing = [c for c in REQUIRED_COLUMNS if c not in rdf.columns]
if missing:
    raise SystemExit(
        "\n\nSTOPPING -- the loaded CSV is missing columns this script needs:\n"
        f"  {missing}\n\n"
        "Rerun nq_v5_stage2_daily_features.py (latest patch adds size/volume/\n"
        "profile-shape fields) to regenerate the CSV, then rerun this script.\n"
    )

BASE_STATS = outcome_stats(rdf, min_n=1)
BASE_BULL_PCT = BASE_STATS['bull_pct'] if BASE_STATS else 50.0
print(f"  Baseline post-IB direction: Bull {BASE_BULL_PCT:.1f}%  (n={BASE_STATS['n'] if BASE_STATS else 0})")

IB_BASE_STATS = outcome_stats(rdf, direction_col='ib_direction', min_n=1)
BASE_IB_BULL_PCT = IB_BASE_STATS['bull_pct'] if IB_BASE_STATS else 50.0
print(f"  Baseline IB direction: Bull {BASE_IB_BULL_PCT:.1f}%  (n={IB_BASE_STATS['n'] if IB_BASE_STATS else 0})")

# ============================================================
# FIXES FROM STAGE 7 DIAGNOSTICS
# ============================================================
# 1. buy_vol_pct bands recalibrated per-column from the empirical
#    distribution (quantiles), not fixed absolute percentages -- the old
#    40/48/52/60 edges were 4-5x too wide for how tightly this variable
#    actually clusters (IB std was 0.89pp; "Balanced" caught 97% of days).
# 2. Volume bands rebuilt as a TRAILING rolling z-score (60-day window,
#    shifted so today's band never depends on today or the future) instead
#    of a static full-sample z-score. The static version let 2022-2023's
#    volume spike (a bear year and a bull year, roughly 50/50) dominate the
#    "Large/Very Large" bucket, averaging away any real directional tilt --
#    and it had look-ahead baked in besides.

rdf = rdf.sort_values('date').reset_index(drop=True)

ROLL_WINDOW = 60
ROLL_MIN_PERIODS = 20


def zscore_band(z):
    if pd.isna(z):
        return None
    if z < -1.5:
        return 'Very Small'
    if z < -0.5:
        return 'Small'
    if z < 0.5:
        return 'Average'
    if z < 1.5:
        return 'Large'
    return 'Very Large'


def rolling_volume_band(series, window=ROLL_WINDOW, min_periods=ROLL_MIN_PERIODS):
    roll_mean = series.rolling(window, min_periods=min_periods).mean().shift(1)
    roll_std = series.rolling(window, min_periods=min_periods).std().shift(1)
    z = (series - roll_mean) / roll_std
    return z.apply(zscore_band), z


def recalibrated_buy_vol_band(series):
    """Percentile-based bands computed from this column's own empirical
    distribution (roughly a 10/15/50/15/10 split), instead of fixed
    absolute percentage-point cutoffs that don't match its actual spread."""
    p10, p25, p75, p90 = series.quantile([0.10, 0.25, 0.75, 0.90])

    def band(x):
        if pd.isna(x):
            return None
        if x < p10:
            return 'Heavy Sell'
        if x < p25:
            return 'Sell-leaning'
        if x <= p75:
            return 'Balanced'
        if x <= p90:
            return 'Buy-leaning'
        return 'Heavy Buy'
    return series.apply(band)


print_header("VOLUME BAND FIX -- OLD (static, full-sample) VS. NEW (trailing 60-day)")

for prefix in ['orb15', 'orb30', 'ib']:
    old_dist = rdf[f'{prefix}_volume_band'].value_counts()
    new_band, new_z = rolling_volume_band(rdf[f'{prefix}_volume'])
    rdf[f'{prefix}_volume_rollband'] = new_band
    rdf[f'{prefix}_volume_rollzscore'] = new_z
    new_dist = new_band.value_counts()
    print_subheader(f"{prefix} volume: old static band vs. new trailing-60-day band")
    for band in BAND_ORDER:
        print(f"    {band:<12}  old={old_dist.get(band,0):5d}   new={new_dist.get(band,0):5d}")

    # recalibrated buy_vol_pct bands, replacing the fixed-threshold version
    rdf[f'{prefix}_buy_vol_band'] = recalibrated_buy_vol_band(rdf[f'{prefix}_buy_vol_pct'])

print()

# top-half / bottom-half delta, precomputed once for the shape+delta section
rdf['ib_top_delta'] = rdf['delta_q3'] + rdf['delta_q4']
rdf['ib_bottom_delta'] = rdf['delta_q1'] + rdf['delta_q2']
rdf['orb15_top_delta'] = rdf['orb15_d3'] + rdf['orb15_d4']
rdf['orb15_bottom_delta'] = rdf['orb15_d1'] + rdf['orb15_d2']
rdf['orb30_top_delta'] = rdf['orb30_d3'] + rdf['orb30_d4']
rdf['orb30_bottom_delta'] = rdf['orb30_d1'] + rdf['orb30_d2']

Q1_COL = {'orb15': 'orb15_d1', 'orb30': 'orb30_d1', 'ib': 'delta_q1'}
Q4_COL = {'orb15': 'orb15_d4', 'orb30': 'orb30_d4', 'ib': 'delta_q4'}


# ============================================================
# PART A -- SIZE
# ============================================================

print_header("PART A -- SIZE: IS THE ORB/IB RANGE LARGER OR SMALLER THAN NORMAL?")

for label, band_col in [('ORB-15', 'orb15_range_pct_band'), ('ORB-30', 'orb30_range_pct_band'),
                        ('IB', 'ib_range_pct_band')]:
    print_subheader(f"{label} range size-band distribution + outcomes")
    for band in BAND_ORDER:
        print_row(f"{label} = {band}", (rdf[band_col] == band).sum(), total_days)
    print()
    for band in BAND_ORDER:
        subset = rdf[rdf[band_col] == band]
        print_outcome(f"{label}={band} -> Post-IB", subset)
        print_break(f"{label}={band}", subset)
        collect_edge(f"{label}_size={band}", subset, BASE_BULL_PCT)

print_subheader("IB size band x close-half x extreme-first (does size change the earlier pattern?)")
for band in ['Small', 'Average', 'Large']:
    band_sub = rdf[rdf['ib_range_pct_band'] == band]
    print(f"\n  IB size = {band} (n={len(band_sub)})")
    for ch in ['upper', 'lower']:
        for ext in ['high', 'low']:
            subset = band_sub[(band_sub['ib_close_half'] == ch) & (band_sub['ib_extreme_first'] == ext)]
            print_break(f"    ib={ch}+ext={ext}", subset)
            collect_edge(f"size={band}+ib={ch}+ext={ext}", subset, BASE_BULL_PCT)


# ============================================================
# PART B -- VOLUME
# ============================================================

print_header("PART B -- VOLUME: HIGHER/LOWER THAN NORMAL, AND BUY/SELL SKEW")

for label, prefix in [('ORB-15', 'orb15'), ('ORB-30', 'orb30'), ('IB', 'ib')]:
    band_col = f'{prefix}_volume_rollband'
    print_subheader(f"{label} volume band (trailing 60-day) distribution + outcomes")
    for band in BAND_ORDER:
        print_row(f"{label} volume = {band}", (rdf[band_col] == band).sum(), total_days)
    print()
    for band in BAND_ORDER:
        subset = rdf[rdf[band_col] == band]
        print_outcome(f"{label} volume={band}", subset)
        print_break(f"{label} volume={band}", subset)
        collect_edge(f"{label}_volume={band}", subset, BASE_BULL_PCT)

print_subheader("Buy-volume share -- 'is it mostly bullish or bearish volume' (distinct from net delta)")
for label, prefix in [('ORB-15', 'orb15'), ('ORB-30', 'orb30'), ('IB', 'ib')]:
    band_col = f'{prefix}_buy_vol_band'
    print(f"\n  {label}:")
    for band in ['Heavy Sell', 'Sell-leaning', 'Balanced', 'Buy-leaning', 'Heavy Buy']:
        subset = rdf[rdf[band_col] == band]
        print_outcome(f"    {label} buy_vol={band}", subset)
        collect_edge(f"{label}_buyvol={band}", subset, BASE_BULL_PCT)

print_subheader("The interesting case: HIGH volume but BALANCED buy/sell split (churn/absorption)")
for label, prefix, vol_band_col in [('ORB-15', 'orb15', 'orb15_volume_rollband'),
                                    ('ORB-30', 'orb30', 'orb30_volume_rollband'),
                                    ('IB', 'ib', 'ib_volume_rollband')]:
    subset = rdf[(rdf[vol_band_col].isin(['Large', 'Very Large'])) &
                 (rdf[f'{prefix}_buy_vol_band'] == 'Balanced')]
    print_outcome(f"{label} high-volume + balanced buy/sell", subset)
    print_break(f"{label} high-volume + balanced", subset)
    collect_edge(f"{label}_highvol_balanced", subset, BASE_BULL_PCT)


# ============================================================
# PART C -- DELTA/VOLUME PROFILE SHAPE (P / b / B / D)
# ============================================================

print_header("PART C -- DELTA/VOLUME PROFILE SHAPE (P / b / B / D)")
print("""
  P = top-heavy volume, thin bottom tail   (short-covering / breakout-up)
  b = bottom-heavy volume, thin top tail   (long-liquidation / breakout-down)
  D = middle-heavy, balanced               (rotational / normal / two-sided)
  B = extremes-heavy, thin middle          (bimodal / double distribution)
""")

for label, shape_col in [('ORB-15', 'orb15_profile_shape'), ('ORB-30', 'orb30_profile_shape'),
                         ('IB', 'ib_profile_shape')]:
    print_subheader(f"{label} shape distribution + outcomes")
    for shape in SHAPE_ORDER:
        print_row(f"{label} shape={shape}", (rdf[shape_col] == shape).sum(), total_days)
    print()
    for shape in SHAPE_ORDER:
        subset = rdf[rdf[shape_col] == shape]
        print_outcome(f"{label} shape={shape} -> Post-IB", subset)
        print_break(f"{label} shape={shape}", subset)
        collect_edge(f"{label}_shape={shape}", subset, BASE_BULL_PCT)

print_subheader("Shape + dominant-quartile delta sign: continuation vs. exhaustion vs. conviction")

for label, prefix in [('ORB-15', 'orb15'), ('ORB-30', 'orb30'), ('IB', 'ib')]:
    shape_col = f'{prefix}_profile_shape'
    top_col = f'{prefix}_top_delta'
    bottom_col = f'{prefix}_bottom_delta'
    q1c, q4c = Q1_COL[prefix], Q4_COL[prefix]

    p_sub = rdf[rdf[shape_col] == 'P']
    print(f"\n  {label} P-shape (top-heavy volume, n={len(p_sub)}):")
    print_outcome("    top-half net BUYING (continuation-confirming)", p_sub[p_sub[top_col] > 0])
    print_outcome("    top-half net SELLING despite volume (exhaustion warning)", p_sub[p_sub[top_col] <= 0])
    collect_edge(f"{label}_P_top_buy", p_sub[p_sub[top_col] > 0], BASE_BULL_PCT)
    collect_edge(f"{label}_P_top_sell", p_sub[p_sub[top_col] <= 0], BASE_BULL_PCT)

    b_sub = rdf[rdf[shape_col] == 'b']
    print(f"\n  {label} b-shape (bottom-heavy volume, n={len(b_sub)}):")
    print_outcome("    bottom-half net SELLING (continuation-confirming)", b_sub[b_sub[bottom_col] < 0])
    print_outcome("    bottom-half net BUYING despite volume (exhaustion warning)", b_sub[b_sub[bottom_col] >= 0])
    collect_edge(f"{label}_b_bottom_sell", b_sub[b_sub[bottom_col] < 0], BASE_BULL_PCT)
    collect_edge(f"{label}_b_bottom_buy", b_sub[b_sub[bottom_col] >= 0], BASE_BULL_PCT)

    d_sub = rdf[rdf[shape_col] == 'D']
    print(f"\n  {label} D-shape (balanced/rotational, n={len(d_sub)}):")
    print_outcome("    D-shape -> Post-IB", d_sub)
    print_break("    D-shape", d_sub)
    collect_edge(f"{label}_D_shape", d_sub, BASE_BULL_PCT)

    b2_sub = rdf[rdf[shape_col] == 'B']
    agree_sub = b2_sub[np.sign(b2_sub[q1c]) == np.sign(b2_sub[q4c])]
    disagree_sub = b2_sub[np.sign(b2_sub[q1c]) != np.sign(b2_sub[q4c])]
    print(f"\n  {label} B-shape (bimodal, n={len(b2_sub)}):")
    print_outcome("    extremes AGREE in sign (broad conviction)", agree_sub)
    print_outcome("    extremes DISAGREE in sign (fighting/two-sided)", disagree_sub)
    collect_edge(f"{label}_B_agree", agree_sub, BASE_BULL_PCT)
    collect_edge(f"{label}_B_disagree", disagree_sub, BASE_BULL_PCT)


# ============================================================
# PART D -- VOLUME BAND x CLOSE-HALF x EXTREME-FIRST
# ============================================================
# Mirrors what Part A already did for size -- volume was left as a
# standalone comparison against baseline instead of layered onto the
# established close-half x extreme-first break-rate framework. Fixing that.

print_header("PART D -- VOLUME BAND x IB CLOSE-HALF x IB EXTREME-FIRST")
print("  (mirrors Part A's size treatment -- volume as a third factor on the")
print("   break-rate framework, not just standalone vs. baseline)\n")

for label, vol_band_col in [('ORB-15', 'orb15_volume_rollband'), ('ORB-30', 'orb30_volume_rollband'),
                            ('IB', 'ib_volume_rollband')]:
    print_subheader(f"{label} volume band (trailing 60-day) x IB close-half x IB extreme-first")
    for band in ['Small', 'Average', 'Large']:
        band_sub = rdf[rdf[vol_band_col] == band]
        print(f"\n  {label} volume = {band} (n={len(band_sub)})")
        for ch in ['upper', 'lower']:
            for ext in ['high', 'low']:
                subset = band_sub[(band_sub['ib_close_half'] == ch) & (band_sub['ib_extreme_first'] == ext)]
                print_break(f"    ib={ch}+ext={ext}", subset)
                collect_edge(f"{label}_vol={band}+ib={ch}+ext={ext}", subset, BASE_BULL_PCT)

print_subheader("IB buy-volume share (skew) x IB close-half x IB extreme-first")
for band in ['Sell-leaning', 'Balanced', 'Buy-leaning']:
    band_sub = rdf[rdf['ib_buy_vol_band'] == band]
    print(f"\n  IB buy_vol = {band} (n={len(band_sub)})")
    for ch in ['upper', 'lower']:
        for ext in ['high', 'low']:
            subset = band_sub[(band_sub['ib_close_half'] == ch) & (band_sub['ib_extreme_first'] == ext)]
            print_break(f"    ib={ch}+ext={ext}", subset)
            collect_edge(f"ib_buyvol={band}+ib={ch}+ext={ext}", subset, BASE_BULL_PCT)


# ============================================================
# PART E -- DOES SIZE/VOLUME/SHAPE SHARPEN THE ORB-30 TRADING-FRAMEWORK SIGNAL?
# ============================================================
# Same test as Stage 5 ran for VWAP: Stage 3's strongest live-safe finding
# was ORB-30 close-half agreeing with its own internal extreme-first order
# -> ~82-83% directional read on IB direction. Does adding size/volume/shape
# as a third factor sharpen that, or just repeat the same information?

print_header("PART E -- DOES SIZE / VOLUME / SHAPE SHARPEN THE ORB-30 TRADING-FRAMEWORK SIGNAL?")
print("  Baseline (Stage 3): ORB-30 close-half x its own internal extreme-first")
print("  -> ~82-83% directional read on IB direction, live-safe at 10:00am.\n")

print_subheader("+ ORB-30 size band")
for band in ['Small', 'Average', 'Large']:
    for ch in ['upper', 'lower']:
        for ext in ['high', 'low']:
            subset = rdf[(rdf['orb30_range_pct_band'] == band) & (rdf['orb30_close_half'] == ch) &
                         (rdf['orb30_internal_extreme_first'] == ext)]
            label = f"orb30_size={band}+ch={ch}+ext={ext}"
            print_outcome(label, subset, direction_col='ib_direction')
            collect_edge(label, subset, BASE_IB_BULL_PCT, direction_col='ib_direction')

print_subheader("+ ORB-30 volume band (trailing 60-day)")
for band in ['Small', 'Average', 'Large']:
    for ch in ['upper', 'lower']:
        for ext in ['high', 'low']:
            subset = rdf[(rdf['orb30_volume_rollband'] == band) & (rdf['orb30_close_half'] == ch) &
                         (rdf['orb30_internal_extreme_first'] == ext)]
            label = f"orb30_vol={band}+ch={ch}+ext={ext}"
            print_outcome(label, subset, direction_col='ib_direction')
            collect_edge(label, subset, BASE_IB_BULL_PCT, direction_col='ib_direction')

print_subheader("+ ORB-30 buy-volume share (skew, not magnitude)")
for band in ['Heavy Sell', 'Sell-leaning', 'Balanced', 'Buy-leaning', 'Heavy Buy']:
    for ch in ['upper', 'lower']:
        for ext in ['high', 'low']:
            subset = rdf[(rdf['orb30_buy_vol_band'] == band) & (rdf['orb30_close_half'] == ch) &
                         (rdf['orb30_internal_extreme_first'] == ext)]
            label = f"orb30_buyvol={band}+ch={ch}+ext={ext}"
            print_outcome(label, subset, direction_col='ib_direction')
            collect_edge(label, subset, BASE_IB_BULL_PCT, direction_col='ib_direction')

print_subheader("+ ORB-30 profile shape")
for shape in SHAPE_ORDER:
    for ch in ['upper', 'lower']:
        for ext in ['high', 'low']:
            subset = rdf[(rdf['orb30_profile_shape'] == shape) & (rdf['orb30_close_half'] == ch) &
                         (rdf['orb30_internal_extreme_first'] == ext)]
            label = f"orb30_shape={shape}+ch={ch}+ext={ext}"
            print_outcome(label, subset, direction_col='ib_direction')
            collect_edge(label, subset, BASE_IB_BULL_PCT, direction_col='ib_direction')

print_subheader("Baseline for comparison (no size/volume/shape)")
for ch in ['upper', 'lower']:
    for ext in ['high', 'low']:
        subset = rdf[(rdf['orb30_close_half'] == ch) & (rdf['orb30_internal_extreme_first'] == ext)]
        print_outcome(f"orb30={ch}+ext={ext}  (no size/vol/shape)", subset, direction_col='ib_direction')


# ============================================================
# RANKED EDGE TABLE
# ============================================================

print_header("RANKED EDGE TABLE -- SIZE / VOLUME / PROFILE-SHAPE PATTERNS, SORTED BY EDGE")
print(f"  Baseline Bull%: {BASE_BULL_PCT:.1f}  |  min n={MIN_DISPLAY_SAMPLE}\n")

edge_df = pd.DataFrame(edge_rows).drop_duplicates(subset=['label']).sort_values('edge_vs_base', ascending=False)
print(f"  {'PATTERN':<55}{'n':>6}{'BULL%':>8}{'BEAR%':>8}{'EDGE':>8}{'BIAS':>6}")
for _, row in edge_df.iterrows():
    print(f"  {row['label']:<55}{row['n']:>6}{row['bull_pct']:>7.1f}%{row['bear_pct']:>7.1f}%"
          f"{row['edge_vs_base']:>7.1f}%{row['bias']:>6}")

edge_df.to_csv(RANKED_EDGE_PATH, index=False)
print(f"\nSaved -> {RANKED_EDGE_PATH}")

print_header("ANALYSIS COMPLETE")

Loading unified dataset...
  1,627 days loaded
  Baseline post-IB direction: Bull 55.7%  (n=1627)
  Baseline IB direction: Bull 52.0%  (n=1627)

             VOLUME BAND FIX -- OLD (static, full-sample) VS. NEW (trailing 60-day)            

-----------------------------------------------------------------------------------------------
  orb15 volume: old static band vs. new trailing-60-day band
-----------------------------------------------------------------------------------------------
    Very Small    old=   86   new=   93
    Small         old=  361   new=  330
    Average       old=  714   new=  714
    Large         old=  367   new=  358
    Very Large    old=   99   new=  112

-----------------------------------------------------------------------------------------------
  orb30 volume: old static band vs. new trailing-60-day band
-----------------------------------------------------------------------------------------------
    Very Small    old=   83   new=   87
    Small  

    ORB-15 volume=Average                         n= 714  Bull: 58.1%  Bear: 41.9%  Edge: 8.1%  [BULL]
    ORB-15 volume=Average                         n= 714  H1h= 39.6%  L1h= 37.1%  Hses= 61.8%  Lses= 55.3%
    ORB-15 volume=Large                           n= 358  Bull: 52.5%  Bear: 47.5%  Edge: 2.5%  [BULL]
    ORB-15 volume=Large                           n= 358  H1h= 38.0%  L1h= 36.9%  Hses= 60.1%  Lses= 55.0%
    ORB-15 volume=Very Large                      n= 112  Bull: 58.0%  Bear: 42.0%  Edge: 8.0%  [BULL]
    ORB-15 volume=Very Large                      n= 112  H1h= 43.8%  L1h= 29.5%  Hses= 60.7%  Lses= 49.1%

-----------------------------------------------------------------------------------------------
  ORB-30 volume band (trailing 60-day) distribution + outcomes
-----------------------------------------------------------------------------------------------
    ORB-30 volume = Very Small                       87  (  5.3%)  ##
    ORB-30 volume = Small                   


  ORB-30 volume = Average (n=732)
        ib=upper+ext=high                         n=  67  H1h= 43.3%  L1h= 17.9%  Hses= 79.1%  Lses= 46.3%
        ib=upper+ext=low                          n= 320  H1h= 71.6%  L1h= 11.2%  Hses= 86.2%  Lses= 32.5%
        ib=lower+ext=high                         n= 292  H1h=  9.9%  L1h= 68.2%  Hses= 33.2%  Lses= 81.2%
        ib=lower+ext=low                          n=  53  H1h= 15.1%  L1h= 58.5%  Hses= 58.5%  Lses= 73.6%

  ORB-30 volume = Large (n=347)
        ib=upper+ext=high                         n=  34  H1h= 35.3%  L1h= 20.6%  Hses= 64.7%  Lses= 52.9%
        ib=upper+ext=low                          n= 140  H1h= 67.9%  L1h=  9.3%  Hses= 80.7%  Lses= 30.7%
        ib=lower+ext=high                         n= 146  H1h=  7.5%  L1h= 65.8%  Hses= 33.6%  Lses= 79.5%
        ib=lower+ext=low                          n=  27  (below min sample)

-----------------------------------------------------------------------------------------------
  IB volu

    orb30=upper+ext=high  (no size/vol/shape)     n= 188  Bull: 52.1%  Bear: 47.9%  Edge: 2.1%  [BULL]
    orb30=upper+ext=low  (no size/vol/shape)      n= 713  Bull: 82.3%  Bear: 17.7%  Edge:32.3%  [BULL]
    orb30=lower+ext=high  (no size/vol/shape)     n= 606  Bull: 16.8%  Bear: 83.2%  Edge:33.2%  [BEAR]
    orb30=lower+ext=low  (no size/vol/shape)      n= 117  Bull: 48.7%  Bear: 51.3%  Edge: 1.3%  [BEAR]

          RANKED EDGE TABLE -- SIZE / VOLUME / PROFILE-SHAPE PATTERNS, SORTED BY EDGE          
  Baseline Bull%: 55.7  |  min n=30

  PATTERN                                                     n   BULL%   BEAR%    EDGE  BIAS
  orb30_buyvol=Heavy Sell+ch=lower+ext=high                 135    3.7%   96.3%   48.3%  BEAR
  orb30_size=Large+ch=lower+ext=high                        111   10.8%   89.2%   41.2%  BEAR
  orb30_vol=Large+ch=lower+ext=high                         152   12.5%   87.5%   39.5%  BEAR
  orb30_buyvol=Heavy Buy+ch=upper+ext=low                   153   90.2%    9.8

In [21]:
"""
NQ IB / ORB / Order-Flow Pipeline -- v5
Stage 5: VWAP anchors.

Four anchors:
  weekly    -- running VWAP anchored Sunday 18:00 ET, sampled at ORB-15/
               ORB-30/IB close (9:45/10:00/10:30)
  session   -- running VWAP anchored 9:30 ET (classic day-VWAP), sampled at
               the same three checkpoints
  prevweek  -- static: last week's whole-week VWAP, carried forward as a
               fixed reference level for every day in the current week
  premarket -- static: the 6:00-9:00am window's VWAP, a fixed reference
               level known before the IB even starts

Part A: distribution + outcomes for each anchor x each price point.
Part B: session-pattern breakdown for the two "live" anchors (weekly, session).
Part C: the actual test -- does adding VWAP position as a third condition
        sharpen the strongest finding from Stage 3 (ORB-30 close-half x its
        own internal extreme-first, live-safe at 10:00am)?
"""

import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CONFIG
# ============================================================

DATA_DIR = Path(r"C:\Users\tammm\NQ and ES analysis")
DATASET_PATH = DATA_DIR / "nq_v5_unified_dataset.csv"
RANKED_EDGE_PATH = DATA_DIR / "nq_v5_vwap_edge_table.csv"

MIN_DISPLAY_SAMPLE = 30

ANCHORS = [
    ('weekly',    'Weekly (Sun 18:00 anchor, running)'),
    ('session',   'Session (9:30 anchor, running -- classic day-VWAP)'),
    ('prevweek',  "Previous week's VWAP (static)"),
    ('premarket', '6-9am premarket VWAP (static)'),
]
PRICE_POINTS = [('ORB-15', 'orb15'), ('ORB-30', 'orb30'), ('IB', 'ib')]

# ============================================================
# HELPERS (same conventions as Stage 3/4)
# ============================================================

def safe_pct(n, d):
    return (n / d * 100) if d > 0 else 0.0


def print_header(text, width=95):
    print("\n" + "=" * width)
    print(text.center(width))
    print("=" * width)


def print_subheader(text, width=95):
    print("\n" + "-" * width)
    print(f"  {text}")
    print("-" * width)


def print_row(label, count, total, indent=4):
    pct = safe_pct(count, total)
    bar = "#" * int(pct / 2)
    print(f"{' '*indent}{label:<50} {count:5d}  ({pct:5.1f}%)  {bar}")


def outcome_stats(subset, direction_col='post_ib_direction', min_n=MIN_DISPLAY_SAMPLE):
    bull = (subset[direction_col] == 'bullish').sum()
    bear = (subset[direction_col] == 'bearish').sum()
    n = bull + bear
    if n < min_n:
        return None
    bull_pct = safe_pct(bull, n)
    return {'n': n, 'bull_pct': bull_pct, 'bear_pct': 100 - bull_pct,
            'edge_vs_50': abs(bull_pct - 50), 'bias': 'BULL' if bull_pct > 50 else 'BEAR'}


def print_outcome(label, subset, indent=4, min_n=MIN_DISPLAY_SAMPLE, direction_col='post_ib_direction'):
    s = outcome_stats(subset, direction_col=direction_col, min_n=min_n)
    if s is None:
        print(f"{' '*indent}{label:<50} n={len(subset):4d}  (below min sample)")
        return
    print(f"{' '*indent}{label:<50} n={s['n']:4d}  "
          f"Bull:{s['bull_pct']:5.1f}%  Bear:{s['bear_pct']:5.1f}%  Edge:{s['edge_vs_50']:4.1f}%  [{s['bias']}]")


def break_stats(subset, min_n=MIN_DISPLAY_SAMPLE):
    n = len(subset)
    if n < min_n:
        return None
    hb1 = subset['ib_high_broken_1h'].sum()
    lb1 = subset['ib_low_broken_1h'].sum()
    hbs = subset['ib_high_broken_session'].sum()
    lbs = subset['ib_low_broken_session'].sum()
    return {'n': n, 'h1h': safe_pct(hb1, n), 'l1h': safe_pct(lb1, n),
            'hses': safe_pct(hbs, n), 'lses': safe_pct(lbs, n)}


def print_break(label, subset, indent=4, min_n=MIN_DISPLAY_SAMPLE):
    s = break_stats(subset, min_n=min_n)
    if s is None:
        print(f"{' '*indent}{label:<50} n={len(subset):4d}  (below min sample)")
        return
    print(f"{' '*indent}{label:<50} n={s['n']:4d}  "
          f"H1h={s['h1h']:5.1f}%  L1h={s['l1h']:5.1f}%  Hses={s['hses']:5.1f}%  Lses={s['lses']:5.1f}%")


edge_rows = []


def collect_edge(label, subset, base_bull_pct, min_n=MIN_DISPLAY_SAMPLE, direction_col='post_ib_direction'):
    s = outcome_stats(subset, direction_col=direction_col, min_n=min_n)
    if s is None:
        return
    edge_rows.append({'label': label, 'n': s['n'], 'bull_pct': s['bull_pct'],
                       'bear_pct': s['bear_pct'], 'edge_vs_base': abs(s['bull_pct'] - base_bull_pct),
                       'bias': s['bias']})


# ============================================================
# LOAD
# ============================================================

print("Loading unified dataset...")
rdf = pd.read_csv(DATASET_PATH, parse_dates=['date'])
total_days = len(rdf)
print(f"  {total_days:,} days loaded")

REQUIRED_COLUMNS = (
    ['post_ib_direction', 'ib_direction', 'ib_close_half', 'ib_extreme_first',
     'orb30_close_half', 'orb30_internal_extreme_first', 'asia_london_pattern',
     'ib_high_broken_1h', 'ib_low_broken_1h', 'ib_high_broken_session', 'ib_low_broken_session'] +
    [f'{p}_vs_vwap_{a}' for a, _ in ANCHORS for _, p in PRICE_POINTS]
)
missing = [c for c in REQUIRED_COLUMNS if c not in rdf.columns]
if missing:
    raise SystemExit(
        "\n\nSTOPPING -- the loaded CSV is missing columns this script needs:\n"
        f"  {missing}\n\n"
        "Rerun nq_v5_stage2_daily_features.py (latest patch adds VWAP anchors)\n"
        "to regenerate the CSV, then rerun this script.\n"
    )

BASE_STATS = outcome_stats(rdf, min_n=1)
BASE_BULL_PCT = BASE_STATS['bull_pct'] if BASE_STATS else 50.0
print(f"  Baseline post-IB direction: Bull {BASE_BULL_PCT:.1f}%  (n={BASE_STATS['n'] if BASE_STATS else 0})")

IB_BASE_STATS = outcome_stats(rdf, direction_col='ib_direction', min_n=1)
BASE_IB_BULL_PCT = IB_BASE_STATS['bull_pct'] if IB_BASE_STATS else 50.0
print(f"  Baseline IB direction: Bull {BASE_IB_BULL_PCT:.1f}%  (n={IB_BASE_STATS['n'] if IB_BASE_STATS else 0})")


# ============================================================
# PART A -- DISTRIBUTION + OUTCOMES, EACH ANCHOR x EACH PRICE POINT
# ============================================================

print_header("PART A -- POSITION VS. EACH VWAP ANCHOR")

for anchor, anchor_label in ANCHORS:
    print_subheader(anchor_label)
    for label, prefix in PRICE_POINTS:
        col = f'{prefix}_vs_vwap_{anchor}'
        above = (rdf[col] == 'above').sum()
        below = (rdf[col] == 'below').sum()
        valid = above + below
        print(f"\n  {label} vs {anchor} VWAP  (n valid = {valid})")
        for pos in ['above', 'below']:
            subset = rdf[rdf[col] == pos]
            print_outcome(f"    {label} {pos}", subset)
            print_break(f"    {label} {pos}", subset)
            collect_edge(f"{label}_vs_vwap_{anchor}={pos}", subset, BASE_BULL_PCT)


# ============================================================
# PART B -- SESSION-PATTERN BREAKDOWN FOR THE TWO "LIVE" ANCHORS
# ============================================================

print_header("PART B -- VWAP POSITION BY ASIA/LONDON STRUCTURAL PATTERN")
print("  (weekly and session anchors only -- these are the two most commonly")
print("   watched intraday VWAPs)\n")

for pat in ['london_engulfs_asia', 'asia_engulfs_london',
            'london_partially_engulfs_upwards', 'london_partially_engulfs_downwards']:
    pat_sub = rdf[rdf['asia_london_pattern'] == pat]
    n_pat = len(pat_sub)
    print_subheader(f"{pat}  (n={n_pat})")
    if n_pat < MIN_DISPLAY_SAMPLE:
        print("    below min sample")
        continue
    for anchor in ['weekly', 'session']:
        col = f'ib_vs_vwap_{anchor}'
        above = (pat_sub[col] == 'above').sum()
        valid = (pat_sub[col].isin(['above', 'below'])).sum()
        print(f"    IB above {anchor} VWAP: {safe_pct(above, valid):5.1f}%  ({above}/{valid})")
    print()
    for anchor in ['weekly', 'session']:
        col = f'ib_vs_vwap_{anchor}'
        for pos in ['above', 'below']:
            sub2 = pat_sub[pat_sub[col] == pos]
            print_outcome(f"  {pat[:22]} + IB {anchor}={pos}", sub2)
            collect_edge(f"pattern={pat[:22]}+ib_vwap_{anchor}={pos}", sub2, BASE_BULL_PCT)


# ============================================================
# PART C -- DOES VWAP SHARPEN THE STRONGEST EXISTING SIGNAL?
# ============================================================
# Stage 3's strongest live-safe finding: ORB-30 close-half agreeing with its
# own internal extreme-first order gave ~82-83% directional read on IB
# direction, on ~81% of days. Does adding VWAP position as a third factor
# sharpen that further, or just repeat the same information?

print_header("PART C -- DOES VWAP SHARPEN THE ORB-30 TRADING-FRAMEWORK SIGNAL?")
print("  Baseline (Stage 3): ORB-30 close-half x its own internal extreme-first")
print("  -> ~82-83% directional read on IB direction, live-safe at 10:00am.\n")

for anchor in ['weekly', 'session']:
    vwap_col = f'orb30_vs_vwap_{anchor}'
    print_subheader(f"ORB-30 close-half x internal extreme-first x vs. {anchor} VWAP -> IB direction")
    for ch in ['upper', 'lower']:
        for ext in ['high', 'low']:
            for vwap_pos in ['above', 'below']:
                subset = rdf[(rdf['orb30_close_half'] == ch) &
                             (rdf['orb30_internal_extreme_first'] == ext) &
                             (rdf[vwap_col] == vwap_pos)]
                label = f"orb30={ch}+ext={ext}+vwap_{anchor}={vwap_pos}"
                print_outcome(label, subset, direction_col='ib_direction')
                collect_edge(label, subset, BASE_IB_BULL_PCT, direction_col='ib_direction')

print_subheader("For comparison: the two-factor baseline without VWAP (from Stage 3)")
for ch in ['upper', 'lower']:
    for ext in ['high', 'low']:
        subset = rdf[(rdf['orb30_close_half'] == ch) & (rdf['orb30_internal_extreme_first'] == ext)]
        print_outcome(f"orb30={ch}+ext={ext}  (no VWAP)", subset, direction_col='ib_direction')


# ============================================================
# RANKED EDGE TABLE
# ============================================================

print_header("RANKED EDGE TABLE -- VWAP PATTERNS, SORTED BY EDGE")
print(f"  Baseline Bull%: {BASE_BULL_PCT:.1f}  |  min n={MIN_DISPLAY_SAMPLE}\n")

edge_df = pd.DataFrame(edge_rows).drop_duplicates(subset=['label']).sort_values('edge_vs_base', ascending=False)
print(f"  {'PATTERN':<58}{'n':>6}{'BULL%':>8}{'BEAR%':>8}{'EDGE':>8}{'BIAS':>6}")
for _, row in edge_df.iterrows():
    print(f"  {row['label']:<58}{row['n']:>6}{row['bull_pct']:>7.1f}%{row['bear_pct']:>7.1f}%"
          f"{row['edge_vs_base']:>7.1f}%{row['bias']:>6}")

edge_df.to_csv(RANKED_EDGE_PATH, index=False)
print(f"\nSaved -> {RANKED_EDGE_PATH}")

print_header("ANALYSIS COMPLETE")

Loading unified dataset...
  1,627 days loaded
  Baseline post-IB direction: Bull 55.7%  (n=1627)
  Baseline IB direction: Bull 52.0%  (n=1627)

                            PART A -- POSITION VS. EACH VWAP ANCHOR                            

-----------------------------------------------------------------------------------------------
  Weekly (Sun 18:00 anchor, running)
-----------------------------------------------------------------------------------------------

  ORB-15 vs weekly VWAP  (n valid = 1627)
        ORB-15 above                                   n= 974  Bull: 57.4%  Bear: 42.6%  Edge: 7.4%  [BULL]
        ORB-15 above                                   n= 974  H1h= 43.9%  L1h= 34.5%  Hses= 64.6%  Lses= 53.1%
        ORB-15 below                                   n= 653  Bull: 53.1%  Bear: 46.9%  Edge: 3.1%  [BULL]
        ORB-15 below                                   n= 653  H1h= 36.9%  L1h= 38.1%  Hses= 59.3%  Lses= 59.0%

  ORB-30 vs weekly VWAP  (n valid = 1627)
   

In [25]:
"""
NQ IB / ORB / Order-Flow Pipeline -- v5
Stage 6: volume/delta profile of the 6-9am premarket session.

Same treatment Stage 4 gave ORB-15/ORB-30/IB, now applied to the premarket
window: size, volume + buy/sell skew, P/b/B/D quartile shape, then whether
premarket direction concords with IB direction, and whether any of this
sharpens the ORB-30 trading-framework signal (the same test run for size/
volume/shape in Stage 4 and for VWAP in Stage 5).
"""

import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CONFIG
# ============================================================

DATA_DIR = Path(r"C:\Users\tammm\NQ and ES analysis")
DATASET_PATH = DATA_DIR / "nq_v5_unified_dataset.csv"
RANKED_EDGE_PATH = DATA_DIR / "nq_v5_premarket_edge_table.csv"

MIN_DISPLAY_SAMPLE = 30
BAND_ORDER = ['Very Small', 'Small', 'Average', 'Large', 'Very Large']
SHAPE_ORDER = ['P', 'b', 'D', 'B']

# ============================================================
# HELPERS (same conventions as Stage 3/4/5)
# ============================================================

def safe_pct(n, d):
    return (n / d * 100) if d > 0 else 0.0


def print_header(text, width=95):
    print("\n" + "=" * width)
    print(text.center(width))
    print("=" * width)


def print_subheader(text, width=95):
    print("\n" + "-" * width)
    print(f"  {text}")
    print("-" * width)


def print_row(label, count, total, indent=4):
    pct = safe_pct(count, total)
    bar = "#" * int(pct / 2)
    print(f"{' '*indent}{label:<45} {count:5d}  ({pct:5.1f}%)  {bar}")


def outcome_stats(subset, direction_col='post_ib_direction', min_n=MIN_DISPLAY_SAMPLE):
    bull = (subset[direction_col] == 'bullish').sum()
    bear = (subset[direction_col] == 'bearish').sum()
    n = bull + bear
    if n < min_n:
        return None
    bull_pct = safe_pct(bull, n)
    return {'n': n, 'bull_pct': bull_pct, 'bear_pct': 100 - bull_pct,
            'edge_vs_50': abs(bull_pct - 50), 'bias': 'BULL' if bull_pct > 50 else 'BEAR'}


def print_outcome(label, subset, indent=4, min_n=MIN_DISPLAY_SAMPLE, direction_col='post_ib_direction'):
    s = outcome_stats(subset, direction_col=direction_col, min_n=min_n)
    if s is None:
        print(f"{' '*indent}{label:<45} n={len(subset):4d}  (below min sample)")
        return
    print(f"{' '*indent}{label:<45} n={s['n']:4d}  "
          f"Bull:{s['bull_pct']:5.1f}%  Bear:{s['bear_pct']:5.1f}%  Edge:{s['edge_vs_50']:4.1f}%  [{s['bias']}]")


def break_stats(subset, min_n=MIN_DISPLAY_SAMPLE):
    n = len(subset)
    if n < min_n:
        return None
    hb1 = subset['ib_high_broken_1h'].sum()
    lb1 = subset['ib_low_broken_1h'].sum()
    hbs = subset['ib_high_broken_session'].sum()
    lbs = subset['ib_low_broken_session'].sum()
    return {'n': n, 'h1h': safe_pct(hb1, n), 'l1h': safe_pct(lb1, n),
            'hses': safe_pct(hbs, n), 'lses': safe_pct(lbs, n)}


def print_break(label, subset, indent=4, min_n=MIN_DISPLAY_SAMPLE):
    s = break_stats(subset, min_n=min_n)
    if s is None:
        print(f"{' '*indent}{label:<45} n={len(subset):4d}  (below min sample)")
        return
    print(f"{' '*indent}{label:<45} n={s['n']:4d}  "
          f"H1h={s['h1h']:5.1f}%  L1h={s['l1h']:5.1f}%  Hses={s['hses']:5.1f}%  Lses={s['lses']:5.1f}%")


edge_rows = []


def collect_edge(label, subset, base_bull_pct, min_n=MIN_DISPLAY_SAMPLE, direction_col='post_ib_direction'):
    s = outcome_stats(subset, direction_col=direction_col, min_n=min_n)
    if s is None:
        return
    edge_rows.append({'label': label, 'n': s['n'], 'bull_pct': s['bull_pct'],
                       'bear_pct': s['bear_pct'], 'edge_vs_base': abs(s['bull_pct'] - base_bull_pct),
                       'bias': s['bias']})


def buy_vol_band(pct):
    if pd.isna(pct):
        return None
    if pct < 40:
        return 'Heavy Sell'
    if pct < 48:
        return 'Sell-leaning'
    if pct < 52:
        return 'Balanced'
    if pct < 60:
        return 'Buy-leaning'
    return 'Heavy Buy'


# ============================================================
# LOAD
# ============================================================

print("Loading unified dataset...")
rdf = pd.read_csv(DATASET_PATH, parse_dates=['date'])
total_days = len(rdf)
print(f"  {total_days:,} days loaded")

REQUIRED_COLUMNS = [
    'premarket_range_pct_band', 'premarket_volume_band', 'premarket_buy_vol',
    'premarket_sell_vol', 'premarket_profile_shape', 'premarket_direction',
    'premarket_close_half', 'premarket_delta_bias',
    'premarket_d1', 'premarket_d2', 'premarket_d3', 'premarket_d4',
    'ib_direction', 'ib_close_half', 'ib_extreme_first',
    'orb30_close_half', 'orb30_internal_extreme_first',
    'ib_high_broken_1h', 'ib_low_broken_1h',
    'ib_high_broken_session', 'ib_low_broken_session', 'post_ib_direction',
]
missing = [c for c in REQUIRED_COLUMNS if c not in rdf.columns]
if missing:
    raise SystemExit(
        "\n\nSTOPPING -- the loaded CSV is missing columns this script needs:\n"
        f"  {missing}\n\n"
        "Rerun nq_v5_stage2_daily_features.py (latest patch adds the premarket\n"
        "size/volume/profile fields) to regenerate the CSV, then rerun this script.\n"
    )

BASE_STATS = outcome_stats(rdf, min_n=1)
BASE_BULL_PCT = BASE_STATS['bull_pct'] if BASE_STATS else 50.0
print(f"  Baseline post-IB direction: Bull {BASE_BULL_PCT:.1f}%  (n={BASE_STATS['n'] if BASE_STATS else 0})")

IB_BASE_STATS = outcome_stats(rdf, direction_col='ib_direction', min_n=1)
BASE_IB_BULL_PCT = IB_BASE_STATS['bull_pct'] if IB_BASE_STATS else 50.0
print(f"  Baseline IB direction: Bull {BASE_IB_BULL_PCT:.1f}%  (n={IB_BASE_STATS['n'] if IB_BASE_STATS else 0})")

rdf['premarket_buy_vol_band'] = rdf['premarket_buy_vol_pct'].apply(buy_vol_band)
rdf['premarket_top_delta'] = rdf['premarket_d3'] + rdf['premarket_d4']
rdf['premarket_bottom_delta'] = rdf['premarket_d1'] + rdf['premarket_d2']


# ============================================================
# PART A -- SIZE
# ============================================================

print_header("PART A -- PREMARKET (6-9AM) SIZE: LARGER OR SMALLER THAN NORMAL?")

print_subheader("Premarket range size-band distribution + outcomes")
for band in BAND_ORDER:
    print_row(f"premarket = {band}", (rdf['premarket_range_pct_band'] == band).sum(), total_days)
print()
for band in BAND_ORDER:
    subset = rdf[rdf['premarket_range_pct_band'] == band]
    print_outcome(f"premarket={band} -> Post-IB", subset)
    print_break(f"premarket={band}", subset)
    collect_edge(f"premarket_size={band}", subset, BASE_BULL_PCT)

print_subheader("Premarket size band x IB close-half x IB extreme-first")
for band in ['Small', 'Average', 'Large']:
    band_sub = rdf[rdf['premarket_range_pct_band'] == band]
    print(f"\n  premarket size = {band} (n={len(band_sub)})")
    for ch in ['upper', 'lower']:
        for ext in ['high', 'low']:
            subset = band_sub[(band_sub['ib_close_half'] == ch) & (band_sub['ib_extreme_first'] == ext)]
            print_break(f"    ib={ch}+ext={ext}", subset)
            collect_edge(f"premarket_size={band}+ib={ch}+ext={ext}", subset, BASE_BULL_PCT)


# ============================================================
# PART B -- VOLUME
# ============================================================

print_header("PART B -- PREMARKET VOLUME: HIGHER/LOWER THAN NORMAL, AND BUY/SELL SKEW")

print_subheader("Premarket volume band distribution + outcomes")
for band in BAND_ORDER:
    print_row(f"premarket volume = {band}", (rdf['premarket_volume_band'] == band).sum(), total_days)
print()
for band in BAND_ORDER:
    subset = rdf[rdf['premarket_volume_band'] == band]
    print_outcome(f"premarket volume={band}", subset)
    print_break(f"premarket volume={band}", subset)
    collect_edge(f"premarket_volume={band}", subset, BASE_BULL_PCT)

print_subheader("Premarket buy-volume share -- 'is premarket flow mostly bullish or bearish'")
for band in ['Heavy Sell', 'Sell-leaning', 'Balanced', 'Buy-leaning', 'Heavy Buy']:
    subset = rdf[rdf['premarket_buy_vol_band'] == band]
    print_outcome(f"premarket buy_vol={band}", subset)
    collect_edge(f"premarket_buyvol={band}", subset, BASE_BULL_PCT)

print_subheader("Premarket delta bias -- distribution + outcomes")
for bias in ['buy', 'sell']:
    subset = rdf[rdf['premarket_delta_bias'] == bias]
    print_row(f"premarket_delta_bias={bias}", len(subset), total_days)
for bias in ['buy', 'sell']:
    subset = rdf[rdf['premarket_delta_bias'] == bias]
    print_outcome(f"premarket_delta_bias={bias}", subset)
    print_break(f"premarket_delta_bias={bias}", subset)
    collect_edge(f"premarket_delta_bias={bias}", subset, BASE_BULL_PCT)


# ============================================================
# PART C -- SHAPE
# ============================================================

print_header("PART C -- PREMARKET DELTA/VOLUME PROFILE SHAPE (P / b / B / D)")

print_subheader("Premarket shape distribution + outcomes")
for shape in SHAPE_ORDER:
    print_row(f"premarket shape={shape}", (rdf['premarket_profile_shape'] == shape).sum(), total_days)
print()
for shape in SHAPE_ORDER:
    subset = rdf[rdf['premarket_profile_shape'] == shape]
    print_outcome(f"premarket shape={shape} -> Post-IB", subset)
    print_break(f"premarket shape={shape}", subset)
    collect_edge(f"premarket_shape={shape}", subset, BASE_BULL_PCT)

print_subheader("Shape + dominant-quartile delta sign: continuation vs. exhaustion vs. conviction")

p_sub = rdf[rdf['premarket_profile_shape'] == 'P']
print(f"\n  Premarket P-shape (top-heavy volume, n={len(p_sub)}):")
print_outcome("    top-half net BUYING (continuation-confirming)", p_sub[p_sub['premarket_top_delta'] > 0])
print_outcome("    top-half net SELLING despite volume (exhaustion warning)",
              p_sub[p_sub['premarket_top_delta'] <= 0])
collect_edge("premarket_P_top_buy", p_sub[p_sub['premarket_top_delta'] > 0], BASE_BULL_PCT)
collect_edge("premarket_P_top_sell", p_sub[p_sub['premarket_top_delta'] <= 0], BASE_BULL_PCT)

b_sub = rdf[rdf['premarket_profile_shape'] == 'b']
print(f"\n  Premarket b-shape (bottom-heavy volume, n={len(b_sub)}):")
print_outcome("    bottom-half net SELLING (continuation-confirming)",
              b_sub[b_sub['premarket_bottom_delta'] < 0])
print_outcome("    bottom-half net BUYING despite volume (exhaustion warning)",
              b_sub[b_sub['premarket_bottom_delta'] >= 0])
collect_edge("premarket_b_bottom_sell", b_sub[b_sub['premarket_bottom_delta'] < 0], BASE_BULL_PCT)
collect_edge("premarket_b_bottom_buy", b_sub[b_sub['premarket_bottom_delta'] >= 0], BASE_BULL_PCT)

d_sub = rdf[rdf['premarket_profile_shape'] == 'D']
print(f"\n  Premarket D-shape (balanced/rotational, n={len(d_sub)}):")
print_outcome("    D-shape -> Post-IB", d_sub)
print_break("    D-shape", d_sub)
collect_edge("premarket_D_shape", d_sub, BASE_BULL_PCT)

b2_sub = rdf[rdf['premarket_profile_shape'] == 'B']
agree_sub = b2_sub[np.sign(b2_sub['premarket_d1']) == np.sign(b2_sub['premarket_d4'])]
disagree_sub = b2_sub[np.sign(b2_sub['premarket_d1']) != np.sign(b2_sub['premarket_d4'])]
print(f"\n  Premarket B-shape (bimodal, n={len(b2_sub)}):")
print_outcome("    extremes AGREE in sign (broad conviction)", agree_sub)
print_outcome("    extremes DISAGREE in sign (fighting/two-sided)", disagree_sub)
collect_edge("premarket_B_agree", agree_sub, BASE_BULL_PCT)
collect_edge("premarket_B_disagree", disagree_sub, BASE_BULL_PCT)


# ============================================================
# PART D -- DOES PREMARKET DIRECTION CONCORD WITH IB DIRECTION?
# ============================================================

print_header("PART D -- DOES PREMARKET DIRECTION CONCORD WITH IB DIRECTION? (AND VICE VERSA)")

sub = rdf[rdf['premarket_direction'].notna() & rdf['ib_direction'].notna()]
bb = ((sub['premarket_direction'] == 'bullish') & (sub['ib_direction'] == 'bullish')).sum()
bear_b = ((sub['premarket_direction'] == 'bullish') & (sub['ib_direction'] == 'bearish')).sum()
b_bear = ((sub['premarket_direction'] == 'bearish') & (sub['ib_direction'] == 'bullish')).sum()
bearbear = ((sub['premarket_direction'] == 'bearish') & (sub['ib_direction'] == 'bearish')).sum()
print(f"\n  Raw joint counts (n={len(sub)}):")
print(f"  {'':<20}{'IB bull':>12}{'IB bear':>12}{'Total':>10}")
print(f"  {'premarket bull':<20}{bb:>12}{bear_b:>12}{bb+bear_b:>10}")
print(f"  {'premarket bear':<20}{b_bear:>12}{bearbear:>12}{b_bear+bearbear:>10}")
print(f"  {'Total':<20}{bb+b_bear:>12}{bear_b+bearbear:>12}{len(sub):>10}")

print(f"\n  premarket direction -> IB direction (row %, the tradeable direction, known by 9:00am)")
for val in ['bullish', 'bearish']:
    s = sub[sub['premarket_direction'] == val]
    n = len(s)
    bull = (s['ib_direction'] == 'bullish').sum()
    bear = (s['ib_direction'] == 'bearish').sum()
    print(f"    premarket={val:<8} (n={n:4d})  -> IB bullish {safe_pct(bull,n):5.1f}%  IB bearish {safe_pct(bear,n):5.1f}%")

print(f"\n  IB direction -> premarket direction (column %, diagnostic, not tradeable)")
for val in ['bullish', 'bearish']:
    s = sub[sub['ib_direction'] == val]
    n = len(s)
    bull = (s['premarket_direction'] == 'bullish').sum()
    bear = (s['premarket_direction'] == 'bearish').sum()
    print(f"    IB={val:<8} (n={n:4d})  -> premarket bullish {safe_pct(bull,n):5.1f}%  premarket bearish {safe_pct(bear,n):5.1f}%")


# ============================================================
# PART E -- DOES PREMARKET SHARPEN THE ORB-30 TRADING-FRAMEWORK SIGNAL?
# ============================================================

print_header("PART E -- DOES PREMARKET SIZE/VOLUME/SHAPE/DIRECTION SHARPEN THE ORB-30 SIGNAL?")
print("  Baseline (Stage 3): ORB-30 close-half x its own internal extreme-first")
print("  -> ~82-83% directional read on IB direction, live-safe at 10:00am.\n")

print_subheader("+ premarket direction")
for pm_dir in ['bullish', 'bearish']:
    for ch in ['upper', 'lower']:
        for ext in ['high', 'low']:
            subset = rdf[(rdf['premarket_direction'] == pm_dir) & (rdf['orb30_close_half'] == ch) &
                         (rdf['orb30_internal_extreme_first'] == ext)]
            label = f"premarket_dir={pm_dir}+ch={ch}+ext={ext}"
            print_outcome(label, subset, direction_col='ib_direction')
            collect_edge(label, subset, BASE_IB_BULL_PCT, direction_col='ib_direction')

print_subheader("+ premarket delta bias")
for bias in ['buy', 'sell']:
    for ch in ['upper', 'lower']:
        for ext in ['high', 'low']:
            subset = rdf[(rdf['premarket_delta_bias'] == bias) & (rdf['orb30_close_half'] == ch) &
                         (rdf['orb30_internal_extreme_first'] == ext)]
            label = f"premarket_delta={bias}+ch={ch}+ext={ext}"
            print_outcome(label, subset, direction_col='ib_direction')
            collect_edge(label, subset, BASE_IB_BULL_PCT, direction_col='ib_direction')

print_subheader("+ premarket profile shape")
for shape in SHAPE_ORDER:
    for ch in ['upper', 'lower']:
        for ext in ['high', 'low']:
            subset = rdf[(rdf['premarket_profile_shape'] == shape) & (rdf['orb30_close_half'] == ch) &
                         (rdf['orb30_internal_extreme_first'] == ext)]
            label = f"premarket_shape={shape}+ch={ch}+ext={ext}"
            print_outcome(label, subset, direction_col='ib_direction')
            collect_edge(label, subset, BASE_IB_BULL_PCT, direction_col='ib_direction')

print_subheader("Baseline for comparison (no premarket factor)")
for ch in ['upper', 'lower']:
    for ext in ['high', 'low']:
        subset = rdf[(rdf['orb30_close_half'] == ch) & (rdf['orb30_internal_extreme_first'] == ext)]
        print_outcome(f"orb30={ch}+ext={ext}  (no premarket)", subset, direction_col='ib_direction')


# ============================================================
# RANKED EDGE TABLE
# ============================================================

print_header("RANKED EDGE TABLE -- PREMARKET PATTERNS, SORTED BY EDGE")
print(f"  Baseline Bull%: {BASE_BULL_PCT:.1f}  |  min n={MIN_DISPLAY_SAMPLE}\n")

edge_df = pd.DataFrame(edge_rows).drop_duplicates(subset=['label']).sort_values('edge_vs_base', ascending=False)
print(f"  {'PATTERN':<55}{'n':>6}{'BULL%':>8}{'BEAR%':>8}{'EDGE':>8}{'BIAS':>6}")
for _, row in edge_df.iterrows():
    print(f"  {row['label']:<55}{row['n']:>6}{row['bull_pct']:>7.1f}%{row['bear_pct']:>7.1f}%"
          f"{row['edge_vs_base']:>7.1f}%{row['bias']:>6}")

edge_df.to_csv(RANKED_EDGE_PATH, index=False)
print(f"\nSaved -> {RANKED_EDGE_PATH}")

print_header("ANALYSIS COMPLETE")

Loading unified dataset...
  1,627 days loaded
  Baseline post-IB direction: Bull 55.7%  (n=1627)
  Baseline IB direction: Bull 52.0%  (n=1627)

                PART A -- PREMARKET (6-9AM) SIZE: LARGER OR SMALLER THAN NORMAL?               

-----------------------------------------------------------------------------------------------
  Premarket range size-band distribution + outcomes
-----------------------------------------------------------------------------------------------
    premarket = Very Small                            0  (  0.0%)  
    premarket = Small                               514  ( 31.6%)  ###############
    premarket = Average                             827  ( 50.8%)  #########################
    premarket = Large                               191  ( 11.7%)  #####
    premarket = Very Large                           95  (  5.8%)  ##

    premarket=Very Small -> Post-IB               n=   0  (below min sample)
    premarket=Very Small                         


  Premarket b-shape (bottom-heavy volume, n=415):
        bottom-half net SELLING (continuation-confirming) n= 270  Bull: 54.1%  Bear: 45.9%  Edge: 4.1%  [BULL]
        bottom-half net BUYING despite volume (exhaustion warning) n= 145  Bull: 55.9%  Bear: 44.1%  Edge: 5.9%  [BULL]

  Premarket D-shape (balanced/rotational, n=516):
        D-shape -> Post-IB                        n= 516  Bull: 57.4%  Bear: 42.6%  Edge: 7.4%  [BULL]
        D-shape                                   n= 516  H1h= 43.2%  L1h= 35.3%  Hses= 63.0%  Lses= 52.9%

  Premarket B-shape (bimodal, n=127):
        extremes AGREE in sign (broad conviction) n=  66  Bull: 54.5%  Bear: 45.5%  Edge: 4.5%  [BULL]
        extremes DISAGREE in sign (fighting/two-sided) n=  61  Bull: 50.8%  Bear: 49.2%  Edge: 0.8%  [BULL]

         PART D -- DOES PREMARKET DIRECTION CONCORD WITH IB DIRECTION? (AND VICE VERSA)        

  Raw joint counts (n=1622):
                           IB bull     IB bear     Total
  premarket bull       

In [26]:
"""
NQ IB / ORB / Order-Flow Pipeline -- v5
Stage 7: volume diagnostics.

Investigates why volume-LEVEL measures showed almost no relationship to
IB/ORB formation in Stages 3-6 (unlike net delta, size, or VWAP position).
Tests three specific hypotheses rather than speculating:

  1. Secular trend  -- has raw volume drifted enough across 2020-2026 that
                        a static full-sample z-score confounds "unusual
                        volume" with "which year this is"?
  2. Buy/sell balance -- is buy_vol_pct clustered so tightly around 50% at
                        these window lengths that the 40/48/52/60 band
                        edges just don't have anywhere to put a meaningful
                        minority of days?
  3. Collinearity with size -- does volume mostly just echo range (which DID
                        show some signal) rather than carrying independent
                        information?
"""

import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path(r"C:\Users\tammm\NQ and ES analysis")
DATASET_PATH = DATA_DIR / "nq_v5_unified_dataset.csv"

PREFIXES = [('orb15', 'ORB-15'), ('orb30', 'ORB-30'), ('ib', 'IB'), ('premarket', 'Premarket')]


def print_header(text, width=95):
    print("\n" + "=" * width)
    print(text.center(width))
    print("=" * width)


def print_subheader(text, width=95):
    print("\n" + "-" * width)
    print(f"  {text}")
    print("-" * width)


print("Loading unified dataset...")
rdf = pd.read_csv(DATASET_PATH, parse_dates=['date'])
rdf['year'] = rdf['date'].dt.year
print(f"  {len(rdf):,} days loaded, years {rdf['year'].min()}-{rdf['year'].max()}")


# ============================================================
# 1 -- SECULAR TREND: mean volume by year + band x year crosstab
# ============================================================

print_header("1 -- VOLUME BY YEAR (secular trend check)")

for prefix, label in PREFIXES:
    vol_col = f'{prefix}_volume'
    if vol_col not in rdf.columns:
        continue
    print_subheader(f"{label} volume by year")
    yearly = rdf.groupby('year')[vol_col].agg(['count', 'mean', 'median'])
    yearly['mean'] = yearly['mean'].round(0).astype(int)
    yearly['median'] = yearly['median'].round(0).astype(int)
    print(yearly.to_string())

    band_col = f'{prefix}_volume_band'
    if band_col in rdf.columns:
        print(f"\n  {label} volume band x year (is 'Large'/'Very Large' concentrated in certain years?)")
        ct = pd.crosstab(rdf['year'], rdf[band_col])
        for band in ['Very Small', 'Small', 'Average', 'Large', 'Very Large']:
            if band not in ct.columns:
                ct[band] = 0
        ct = ct[['Very Small', 'Small', 'Average', 'Large', 'Very Large']]
        print(ct.to_string())
        print()


# ============================================================
# 2 -- BUY/SELL BALANCE: raw buy_vol_pct distribution, not banded
# ============================================================

print_header("2 -- BUY-VOLUME SHARE: RAW DISTRIBUTION (not banded)")

for prefix, label in PREFIXES:
    col = f'{prefix}_buy_vol_pct'
    if col not in rdf.columns:
        continue
    s = rdf[col].dropna()
    print_subheader(f"{label} buy_vol_pct distribution (n={len(s)})")
    print(f"  mean = {s.mean():.2f}   std = {s.std():.2f}")
    for p in [1, 5, 10, 25, 50, 75, 90, 95, 99]:
        print(f"    p{p:>2d} = {s.quantile(p/100):6.2f}")
    within_1pct = ((s >= 49) & (s <= 51)).sum()
    within_2pct = ((s >= 48) & (s <= 52)).sum()
    within_5pct = ((s >= 45) & (s <= 55)).sum()
    print(f"\n  within 49-51%: {within_1pct} ({within_1pct/len(s)*100:.1f}%)")
    print(f"  within 48-52%: {within_2pct} ({within_2pct/len(s)*100:.1f}%)  <- current 'Balanced' band")
    print(f"  within 45-55%: {within_5pct} ({within_5pct/len(s)*100:.1f}%)")


# ============================================================
# 3 -- COLLINEARITY: does volume mostly just echo size?
# ============================================================

print_header("3 -- DOES VOLUME MOSTLY JUST ECHO SIZE (RANGE)?")

for prefix, label in [('orb15', 'ORB-15'), ('orb30', 'ORB-30'), ('ib', 'IB')]:
    vol_col = f'{prefix}_volume'
    range_col = f'{prefix}_range_pct'
    if vol_col not in rdf.columns or range_col not in rdf.columns:
        continue
    corr = rdf[[vol_col, range_col]].corr().iloc[0, 1]
    print(f"  {label}: corr(volume, range_pct) = {corr:.3f}")

print_header("DIAGNOSTIC COMPLETE")

Loading unified dataset...
  1,627 days loaded, years 2020-2026

                           1 -- VOLUME BY YEAR (secular trend check)                           

-----------------------------------------------------------------------------------------------
  ORB-15 volume by year
-----------------------------------------------------------------------------------------------
      count   mean  median
year                      
2020    256  37098   35099
2021    258  43232   42798
2022    258  46646   46812
2023    257  47093   47153
2024    259  44189   44084
2025    256  38806   38426
2026     83  41276   40759

  ORB-15 volume band x year (is 'Large'/'Very Large' concentrated in certain years?)
orb15_volume_band  Very Small  Small  Average  Large  Very Large
year                                                            
2020                       23    118       68     33          14
2021                       11     37      145     54          11
2022                       10    